# 0. Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

## 0.1. Helper Functions

In [ ]:
def plot_distribuicao_por_grupos(df_contratado, df_adimplente, df_inadimplente, col, titulo_legenda=None):
    """
    Plota a distribuição de uma variável categórica ao longo do tempo para três grupos.
    Linhas são interrompidas quando os valores são zero e todos os gráficos usam a mesma escala.
    """
    
    # Criar as tabelas para cada grupo
    tabela_contratado = pd.crosstab(df_contratado['ano_mes'], df_contratado[col], normalize='index')
    tabela_adimplente = pd.crosstab(df_adimplente['ano_mes'], df_adimplente[col], normalize='index')
    tabela_inadimplente = pd.crosstab(df_inadimplente['ano_mes'], df_inadimplente[col], normalize='index')
    
    # Substituir valores 0 por NaN para criar descontinuidades nas linhas
    tabela_contratado = tabela_contratado.replace(0, np.nan)
    tabela_adimplente = tabela_adimplente.replace(0, np.nan)
    tabela_inadimplente = tabela_inadimplente.replace(0, np.nan)
    
    # Encontrar o valor máximo entre todas as tabelas para definir a mesma escala
    max_valor = max(
        tabela_contratado.max().max() if not tabela_contratado.empty else 0,
        tabela_adimplente.max().max() if not tabela_adimplente.empty else 0,
        tabela_inadimplente.max().max() if not tabela_inadimplente.empty else 0
    )
    
    # Adicionar uma pequena margem no topo (10% a mais)
    ylim_max = min(max_valor * 1.1, 1.0)  # Não ultrapassar 100%
    
    # Criar figura com 3 subplots lado a lado
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Lista com as tabelas e títulos
    tabelas = [
        (tabela_contratado, 'Contratados Total'),
        (tabela_adimplente, 'Adimplentes'),
        (tabela_inadimplente, 'Inadimplentes')
    ]
    
    # Título geral da figura
    fig.suptitle(f'Distribuição de {col} ao longo do tempo', 
                 fontsize=16)
    
    # Plotar cada gráfico
    for ax, (tabela, titulo) in zip(axes, tabelas):
        tabela.plot(ax=ax, figsize=(18,5), legend=False)


        # Pintar fundo dos diferentes períodos         
        ax.axvspan('2022-03-01', '2023-03-01', color='lightgreen', alpha=0.3)         # Janela de Modelagem
        ax.axvspan('2023-03-01', '2024-03-01', color='lightblue', alpha=0.3)          # Janela de Observação
        ax.axvspan('2024-05-01', tabela.index.max(), color='lightpink', alpha=0.3)    # Em Produção
        
        ax.axvspan('2024-03-01', '2024-05-01', color='black', alpha=0.4)              # Limbo
        ax.axvspan('2023-03-01', '2023-03-01', color='black', alpha=0.4)              # Marcador
        ax.axvspan('2023-09-01', '2023-09-01', color='black', alpha=0.4)              # Efeito Modelo Teste
        ax.axvspan('2025-03-01', '2025-03-01', color='black', alpha=0.4)              # Marcador
        
        ax.axvspan('2024-10-01', '2025-01-01', color='orange', alpha=0.4)             # Efeito Modelo Produção
        ax.axvspan('2025-01-01', tabela.index.max(), color='lightgray', alpha=0.4)    # Não tem target 1
        
        ax.set_title(titulo)
        ax.set_xlabel('Ano-Mês')
        ax.set_ylabel('Proporção %')
        
        # Definir a mesma escala para todos os gráficos
        ax.set_ylim(0, ylim_max)
        
        # Formatar y-axis como porcentagem
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
        
        # Rotacionar labels do eixo x para melhor legibilidade
        ax.tick_params(axis='x', rotation=45)
    
    # Criar uma única legenda para todos os gráficos
    lines = axes[0].get_lines()
    labels = [line.get_label() for line in lines]
    
    # Definir título da legenda
    if titulo_legenda is None:
        titulo_legenda = f'Valores de {col}'
    
    # Posicionar a legenda fora dos gráficos
    fig.legend(lines, labels, 
               loc='center', 
               bbox_to_anchor=(0.5, -0.05),
               ncol=len(labels),
               title=titulo_legenda,
               columnspacing=1.5)
    
    # Ajustar espaçamento para não cortar a legenda
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)
    
    plt.show()
    
    return fig, axes

In [ ]:
def plot_estatisticas_por_tipo(df, col_numerica, titulo=None):
    """
    Plota média, mediana e coeficiente de variação de uma variável numérica 
    ao longo do tempo para cada tipo.
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame com as colunas 'ano_mes', 'tipo' e a coluna numérica
    col_numerica : str
        Nome da coluna numérica a ser analisada
    titulo : str, optional
        Título personalizado para o gráfico
    """
    
    # Calcular estatísticas por ano_mes e tipo
    df_stats = df.groupby(['ano_mes', 'tipo'])[col_numerica].agg([
        ('media', 'mean'),
        ('mediana', 'median'),
        ('std', 'std')
    ]).reset_index()
    
    # Calcular coeficiente de variação (CV = std/mean)
    df_stats['cv'] = df_stats['std'] / df_stats['media']
    
    # Ordenar tipos para consistência nas cores
    ordem_tipos = ['adimplente', 'inadimplente', 'nao_contratado']
    cores = {'nao_contratado': '#95A5A6', 
             'adimplente': '#2ECC71', 
             'inadimplente': '#E74C3C'}
    
    # Criar figura com 3 subplots lado a lado
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Lista com as estatísticas e títulos
    estatisticas = [
        ('media', 'Média'),
        ('mediana', 'Mediana'),
        ('cv', 'Coeficiente de Variação (σ/μ)')
    ]
    
    # Título geral da figura
    titulo_geral = titulo if titulo else f'Estatísticas de {col_numerica} ao longo do tempo'
    fig.suptitle(titulo_geral, fontsize=16)
    
    # Plotar cada gráfico
    for ax, (estatistica, titulo_graf) in zip(axes, estatisticas):
        
        # Plotar linhas para cada tipo
        for tipo in ordem_tipos:
            dados_tipo = df_stats[df_stats['tipo'] == tipo]
            if not dados_tipo.empty:
                ax.plot(dados_tipo['ano_mes'], dados_tipo[estatistica], 
                       'o-', label=tipo, color=cores.get(tipo, '#333333'), markersize=6)
        
        # Pintar fundo dos períodos (mesmos em todos os gráficos)
        ax.axvspan('2022-03-01', '2023-03-01', color='lightgreen', alpha=0.3)  # Janela de Modelagem
        ax.axvspan('2023-03-01', '2024-03-01', color='lightblue', alpha=0.3)   # Janela de Observação
        ax.axvspan('2024-05-01', '2025-12-01', color='lightpink', alpha=0.3)   # Em Produção
        ax.axvspan('2024-03-01', '2024-05-01', color='black', alpha=0.4)       # Limbo
        ax.axvspan('2023-03-01', '2023-03-01', color='black', alpha=0.4)       # Marcador
        ax.axvspan('2023-09-01', '2023-09-01', color='black', alpha=0.4)       # Efeito Modelo Teste
        ax.axvspan('2025-03-01', '2025-03-01', color='black', alpha=0.4)       # Marcador
        ax.axvspan('2024-10-01', '2025-01-01', color='orange', alpha=0.4)      # Efeito Modelo Produção
        ax.axvspan('2025-01-01', '2025-12-01', color='lightgray', alpha=0.4)   # Não tem target 1
        
        # Configurações específicas para CV
        if estatistica == 'cv':
            ax.set_ylabel('Coeficiente de Variação', fontsize=12)
            # Adicionar linha de referência para CV=0.2 (alta variabilidade)
            ax.axhline(y=0.2, color='red', linestyle='--', alpha=0.5, linewidth=1, label='CV=0.2')
        else:
            ax.set_ylabel(f'{titulo_graf} de {col_numerica}', fontsize=12)
        
        ax.set_xlabel('Ano-Mês', fontsize=12)
        ax.set_title(titulo_graf, fontsize=13)
        ax.tick_params(axis='x', rotation=45)
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)  # Ajustar para o título geral
    plt.show()
    
    return fig, axes

In [ ]:
from scipy.stats import ks_2samp

def calcular_ks(y_target, y_feature):
    """
    Calcula a estatística KS entre duas distribuições
    
    Parameters:
    -----------
    y_target : array-like
        Variável target (0 = adimplente, 1 = inadimplente)
    y_feature : array-like
        Feature do modelo (probabilidade ou pontuação)
    """
    # Separar scores por classe
    scores_adimplentes = y_feature[y_target == 0]
    scores_inadimplentes = y_feature[y_target == 1]
    
    # Calcular KS usando scipy
    ks_statistic, p_value = ks_2samp(scores_inadimplentes, scores_adimplentes)
    
    return ks_statistic, p_value

def calcular_ks_manual(y_target, y_feature):
    """
    Versão manual para entender o cálculo
    """
    # Criar DataFrame
    df = pd.DataFrame({'score': y_feature, 'target': y_target})
    df = df.sort_values('score')
    
    # Calcular distribuições acumuladas
    total_adimplentes = (df['target'] == 0).sum()
    total_inadimplentes = (df['target'] == 1).sum()
    
    df['cum_adimplentes'] = (df['target'] == 0).cumsum() / total_adimplentes
    df['cum_inadimplentes'] = (df['target'] == 1).cumsum() / total_inadimplentes
    df['diferenca'] = abs(df['cum_inadimplentes'] - df['cum_adimplentes'])
    
    ks = df['diferenca'].max()
    ponto_ks = df.loc[df['diferenca'].idxmax(), 'score']
    
    return ks, ponto_ks
    
def plot_ks(y_target, y_score, titulo="Curva KS"):
    """
    Plota a curva KS mostrando a separação entre as distribuições
    """
    # Preparar dados
    df = pd.DataFrame({'score': y_score, 'target': y_target})
    
    # Calcular KS
    ks, ponto_ks = calcular_ks_manual(y_target, y_score)
    
    # Plotar
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico 1: Distribuições
    for target, cor, label in [(0, 'green', 'Adimplentes'), (1, 'red', 'Inadimplentes')]:
        sns.kdeplot(data=df[df['target'] == target], x='score', 
                    label=label, color=cor, ax=ax1, fill=True, alpha=0.3)
    
    ax1.axvline(ponto_ks, color='blue', linestyle='--', 
                label=f'Ponto KS: {ponto_ks:.2f}')
    ax1.set_title(f'Distribuições dos Scores\nKS = {ks:.3f}')
    ax1.set_xlabel('Score')
    ax1.set_ylabel('Densidade')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Gráfico 2: Curva KS (acumuladas)
    df_sorted = df.sort_values('score')
    total_adimplentes = (df['target'] == 0).sum()
    total_inadimplentes = (df['target'] == 1).sum()
    
    cum_adimplentes = (df_sorted['target'] == 0).cumsum() / total_adimplentes
    cum_inadimplentes = (df_sorted['target'] == 1).cumsum() / total_inadimplentes
    scores = df_sorted['score'].values
    
    ax2.plot(scores, cum_adimplentes, 'g-', label='Adimplentes (acumulado)')
    ax2.plot(scores, cum_inadimplentes, 'r-', label='Inadimplentes (acumulado)')
    ax2.plot(scores, abs(cum_inadimplentes - cum_adimplentes), 
             'b--', label='Diferença', alpha=0.7)
    
    # Destacar ponto KS
    idx_ks = np.argmax(abs(cum_inadimplentes - cum_adimplentes))
    ax2.scatter(scores[idx_ks], abs(cum_inadimplentes - cum_adimplentes)[idx_ks],
               color='blue', s=100, zorder=5)
    
    ax2.set_title('Curva KS - Distribuições Acumuladas')
    ax2.set_xlabel('Score')
    ax2.set_ylabel('Proporção Acumulada')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(titulo, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return ks

In [ ]:
def calcular_fpd(df, col_contrato='contratado', col_fpd='FPD',
                 col_score='score', score_minimo=4):
    """
    Calcula taxa de FPD (First Payment Default) por faixa de score.

    Filtra apenas contratos efetivados e com score ativo (> score_minimo).

    Returns
    -------
    DataFrame com colunas: score, total_contratos, total_fpd, taxa_fpd
    """
    contratados = df[(df[col_contrato] == 1) & (df[col_score] > score_minimo)].copy()
    contratados['fpd_flag'] = contratados[col_fpd].fillna(0)

    resultado = (contratados.groupby(col_score)
                             .agg(total_contratos=('fpd_flag', 'count'),
                                  total_fpd=('fpd_flag', 'sum'),
                                  taxa_fpd=('fpd_flag', 'mean'))
                             .round(4)
                             .reset_index())
    return resultado

In [ ]:
def calcular_fpd(df, col_contrato='contratado', col_fpd='FPD', col_score='score'):
    """
    Calcula taxa de FPD por faixa de score
    
    Parameters:
    -----------
    df : DataFrame
        Dados com contratos e indicador FPD
    col_contrato : str
        Coluna que indica se o contrato foi efetivado
    col_fpd : str
        Coluna com indicador FPD (0/1 ou NaN para não contratados)
    col_score : str
        Coluna com score do modelo
    """
    
    # Filtrar apenas contratados
    contratados = df[df[col_contrato] == 1].copy()
    
    # Tratar FPD (assumindo que NaN significa não FPD)
    contratados['fpd_flag'] = contratados[col_fpd].fillna(0)
    
    # Calcular taxa de FPD por score
    fpd_por_score = contratados.groupby(col_score).agg({
        'fpd_flag': ['count', 'sum', 'mean']
    }).round(4)
    
    fpd_por_score.columns = ['total_contratos', 'total_fpd', 'taxa_fpd']
    fpd_por_score = fpd_por_score.reset_index()
    
    return fpd_por_score

In [ ]:
def calcular_woe_iv(df, var, target='target'):
    """
    Calcula WOE (Weight of Evidence) e IV (Information Value) para uma variável
    
    Parameters:
    -----------
    df : DataFrame
        Dados com a variável e target
    var : str
        Nome da variável (categórica ou binned)
    target : str
        Nome da coluna target (0 = adimplente, 1 = inadimplente)
    
    Returns:
    --------
    DataFrame com estatísticas e WOE/IV
    """
    
    # Tabela de contingência
    tabela = pd.crosstab(df[var], df[target], margins=True, margins_name='Total')
    
    # Renomear colunas
    tabela.columns = ['Adimplentes', 'Inadimplentes', 'Total']
    
    # Calcular percentuais
    total_adimplentes = tabela.loc['Total', 'Adimplentes']
    total_inadimplentes = tabela.loc['Total', 'Inadimplentes']
    total_geral = tabela.loc['Total', 'Total']
    
    # DataFrame para resultados (excluir linha Total)
    resultado = tabela.drop('Total').copy()
    
    # Calcular percentuais
    resultado['%_Adimplentes'] = resultado['Adimplentes'] / total_adimplentes
    resultado['%_Inadimplentes'] = resultado['Inadimplentes'] / total_inadimplentes
    
    # Calcular WOE (com suavização para evitar log(0))
    resultado['WOE'] = np.log(
        (resultado['%_Inadimplentes'] + 1e-10) / 
        (resultado['%_Adimplentes'] + 1e-10)
    )
    
    # Calcular contribuição para IV
    resultado['IV_contrib'] = (resultado['%_Inadimplentes'] - resultado['%_Adimplentes']) * resultado['WOE']
    
    # IV total
    iv_total = resultado['IV_contrib'].sum()
    
    # Adicionar totais
    resultado.loc['Total'] = [
        total_adimplentes, total_inadimplentes, total_geral,
        1.0, 1.0, np.nan, iv_total
    ]
    
    return resultado, iv_total

def plot_woe_analysis(df, var, target='target', titulo=None):
    """
    Visualiza análise WOE para uma variável
    """
    # Calcular WOE
    woe_df, iv_total = calcular_woe_iv(df, var, target)
    
    # Remover linha Total para plot
    woe_plot = woe_df.drop('Total').copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico 1: Distribuição das classes
    ax1 = axes[0]
    x = range(len(woe_plot.index))
    width = 0.35
    
    ax1.bar([i - width/2 for i in x], woe_plot['%_Adimplentes'], width, 
            label='Adimplentes', color='green', alpha=0.7)
    ax1.bar([i + width/2 for i in x], woe_plot['%_Inadimplentes'], width, 
            label='Inadimplentes', color='red', alpha=0.7)
    
    ax1.set_xlabel(var)
    ax1.set_ylabel('Proporção')
    ax1.set_title('Distribuição por Classe')
    ax1.set_xticks(x)
    ax1.set_xticklabels(woe_plot.index, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Gráfico 2: WOE
    ax2 = axes[1]
    cores = ['red' if w > 0 else 'green' for w in woe_plot['WOE']]
    ax2.bar(x, woe_plot['WOE'], color=cores, alpha=0.7, edgecolor='black')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    # Adicionar valores
    for i, (idx, row) in enumerate(woe_plot.iterrows()):
        ax2.text(i, row['WOE'] + (0.02 if row['WOE'] > 0 else -0.04), 
                f'{row["WOE"]:.2f}', ha='center', fontsize=9)
    
    ax2.set_xlabel(var)
    ax2.set_ylabel('WOE (Weight of Evidence)')
    ax2.set_title(f'Perfil WOE - IV Total: {iv_total:.3f}')
    ax2.set_xticks(x)
    ax2.set_xticklabels(woe_plot.index, rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Interpretação do IV
    if iv_total < 0.02:
        cor_iv = 'red'
        interprete = 'Não preditivo'
    elif iv_total < 0.1:
        cor_iv = 'orange'
        interprete = 'Fraco'
    elif iv_total < 0.3:
        cor_iv = 'blue'
        interprete = 'Médio'
    else:
        cor_iv = 'green'
        interprete = 'Forte'
    
    ax2.text(0.5, -0.15, f'IV = {iv_total:.3f} - Poder preditivo: {interprete}', 
            transform=ax2.transAxes, ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=cor_iv, alpha=0.3))
    
    titulo_geral = titulo if titulo else f'Análise WOE - {var}'
    plt.suptitle(titulo_geral, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return woe_df, iv_total



## 0.2. Loading Data

In [ ]:
# Load do arquivo
path_data  = "../data/"
df = pd.read_csv(path_data + '2_envio_case_base_Total_2026.csv', sep=';', encoding='latin-1')

df.head()

📊 **Visão geral da base de dados carregada**

- **O que representa:** A base contém propostas de financiamento de veículos com informações de data, score, target de inadimplência (12 meses), variáveis cadastrais e comportamentais do cliente.
- **Período coberto:** março/2022 a dezembro/2025 — abrangendo desenvolvimento do modelo, teste fora do tempo e produção.
- **Colunas-chave para monitoramento:**
  - `score`: saída do modelo (quanto maior, melhor crédito)
  - `target`: 1 = inadimplente em 12 meses, 0 = adimplente, NaN = sem maturação suficiente
  - `contratado`: se a proposta foi efetivada (filtra o ponto cego)
  - `FPD`: primeiro pagamento em default — **não é substituto do target** (ver Easter Egg #8)
- **Atenção:** A base foi carregada com `encoding='latin-1'` e separador `;`, o que é esperado para arquivos exportados de sistemas legados brasileiros.

## 0.3. Tratamento de variáveis e Novas colunas

In [ ]:
# Formatação da coluna 'ano_mes' para datetime
df['ano_mes'] = pd.to_datetime(df['ano_mes'], format='%Y%m')

In [ ]:
print('Substituir 1 Registro da coluna vl_bem, de "25a70" para "(25000, 70000]"')

# Substituir valores '25a70' por '(25000, 70000]'
idx = df[df['vl_bem'] == '25a70'].index
df['vl_bem'] = df['vl_bem'].replace('25a70', '(25000, 70000]')

# Verificar se a substituição foi realizada
df[df.index.isin(idx)]

🥚 **Easter Egg identificado: Categoria inválida em `vl_bem`**

- **O que é:** O campo `vl_bem` (valor do bem financiado) contém o valor `'25a70'`, que é uma string inválida resultante provavelmente de um erro de digitação ou exportação do sistema. O valor correto seria a faixa `(25000, 70000]`.
- **Como foi detectado:** Inspeção dos valores únicos da variável categórica `vl_bem` revelou um valor fora do padrão `(mínimo, máximo]` usado nas demais categorias.
- **Implicação para o modelo:** Se mantido na base, esse registro espúrio distorce o cálculo de PSI (Population Stability Index) e WoE/IV para a variável `vl_bem`, podendo criar uma faixa fictícia com comportamento de inadimplência não representativo.
- **Recomendação de tratamento:** Mapear `'25a70'` para `'(25000, 70000]'` (conforme feito nesta célula) ou excluir o registro. Aguiar et al. recomendam tratar dados inválidos antes de qualquer cálculo de métricas de monitoramento.
- **Narrativa para apresentação:** "Um registro com valor de bem digitado incorretamente foi encontrado e corrigido — sem esse ajuste, os cálculos de estabilidade da variável `vl_bem` seriam distorcidos."

In [ ]:
# Coluna para janela de tempo (modelagem, observação, limbo, produção verde, produção madura)
# Definir condições
condicoes = [
    (df['ano_mes'] < '2023-03-01'),                                    # Janela de Modelagem
    (df['ano_mes'] >= '2023-03-01') & (df['ano_mes'] < '2024-03-01'),  # Janela de teste OOT
    (df['ano_mes'] >= '2024-03-01') & (df['ano_mes'] < '2024-05-01'),  # Limbo
    (df['ano_mes'] >= '2024-05-01') & (df['ano_mes'] < '2025-01-01'),  # Em Produção target
    (df['ano_mes'] >= '2025-01-01')                                    # Em Produção contrato
]

# Definir valores correspondentes
valores = [
    'modelagem',
    'observacao',
    'limbo',
    'target_completo',
    'target_vazio'
]

# Criar a coluna janela
df['janela'] = np.select(condicoes, valores, default='Outra')

# Imprimir o total de registros em cada grupo
print("---" ) 
print("QUANTIDADE DE REGISTROS POR TIPO")
print("  1. Janela de Modelagem    : ", f"{len(df[df['janela'] == 'modelagem']):,}", f" | {len(df[df['janela'] == 'modelagem'])/(len(df)):.2%} da base")
print("  2. Janela de OOT          : ", f"{len(df[df['janela'] == 'observacao']):,}", f" | {len(df[df['janela'] == 'observacao'])/(len(df)):.2%} da base")
print("  3. Gap do Limbo           : ", f"{len(df[df['janela'] == 'limbo']):,}", f"  | {len(df[df['janela'] == 'limbo'])/(len(df)):.2%} da base")
print("  4. Janela de Produção     : ", f"{len(df[df['janela'] == 'target_completo']):,} + {len(df[df['janela'] == 'target_vazio']):,}", f"| {len(df[df['janela'] == 'target_completo'])/(len(df)):.2%} dos Contratados")
print("     - Target completo      : ", f"{len(df[df['janela'] == 'target_completo']):,} ", f"| {len(df[df['janela'] == 'target_completo'])/(len(df)):.2%} dos Contratados")
print("     - Target vazio         : ", f"{len(df[df['janela'] == 'target_vazio']):,} ", f"| {len(df[df['janela'] == 'target_vazio'])/(len(df)):.2%} dos Contratados")

# Imprimir o período de análise por grupo
print("\n" "---" ) 
print("PERÍODO DE ANÁLISE POR TIPO")
print(f"  1. Janela de modelagem    :  {df[df['janela'] == 'modelagem']['ano_mes'].min()} a {df[df['janela'] == 'modelagem']['ano_mes'].max()}")
print(f"  2. Janela de observação   :  {df[df['janela'] == 'observacao']['ano_mes'].min()} a {df[df['janela'] == 'observacao']['ano_mes'].max()}")
print(f"  3. Gap do Limbo           :  {df[df['janela'] == 'limbo']['ano_mes'].min()} a {df[df['janela'] == 'limbo']['ano_mes'].max()}")
print(f"  4. Janela de produção     :  {df[df['janela'] == 'target_completo']['ano_mes'].min()} a {df[df['janela'] == 'target_completo']['ano_mes'].max()} e {df[df['janela'] == 'target_vazio']['ano_mes'].min()} a {df[df['janela'] == 'target_vazio']['ano_mes'].max()}")
print(f"     - Target completo      :  {df[df['janela'] == 'target_completo']['ano_mes'].min()} a {df[df['janela'] == 'target_completo']['ano_mes'].max()}")
print(f"     - Target vazio         :  {df[df['janela'] == 'target_vazio']['ano_mes'].min()} a {df[df['janela'] == 'target_vazio']['ano_mes'].max()}")

📊 **Criação das janelas temporais de monitoramento**

**Volumes por janela:**
| Janela | Período | Registros | % da base | Uso no monitoramento |
|--------|---------|-----------|-----------|----------------------|
| Modelagem (DEV) | mar/22 – fev/23 | 239.889 | 28.29% | Referência para PSI, WoE e baseline de inadimplência |
| Observação (OOT) | mar/23 – fev/24 | 239.940 | 28.29% | Teste fora do tempo para discriminação e previsibilidade |
| **Limbo** | **mar/24 – abr/24** | **40.237** | **4.74%** | **EXCLUÍDO — comportamento híbrido** |
| Produção (target) | mai/24 – dez/24 | 159.935 | 18.86% | Target válido: KS, Gini, Obs vs Esp |
| Produção (perfil) | jan/25 – dez/25 | 168.000 | 19.81% | Apenas estabilidade/perfil |

🥚 **Easter Egg #6 — Período de Limbo (mar/24–abr/24):**
- **O que é:** Os 40.237 registros de março e abril/2024 correspondem ao período de transição entre o modelo de teste OOT e a entrada em produção definitiva. Nesse período, o modelo estava sendo operado de forma experimental, com possível mistura de critérios de aprovação antigos e novos.
- **Como foi detectado:** A definição explícita da janela "limbo" no código e a análise temporal revelam uma descontinuidade entre o comportamento OOT (mar/23–fev/24) e o início de produção (mai/24+).
- **Implicação para o modelo:** Incluir o limbo em análises de estabilidade cria uma descontinuidade artificial na série temporal do PSI e da carta de controle. O comportamento dos clientes desse período mistura características de dois regimes diferentes de concessão de crédito.
- **Recomendação:** Excluir o limbo de todas as análises (conforme feito no código) e documentar explicitamente no relatório que esses 40.237 registros (~4.74% da base) foram excluídos por comportamento híbrido.
- **Narrativa para apresentação:** "Dois meses da base foram descartados — eram o período de transição em que a empresa estava testando a nova política de crédito. Misturar esse período com os demais distorceria qualquer análise de tendência."

In [ ]:
# Verificar os valores únicos da coluna 'FPD'  e 'target' para contratados e não contratados
print(df[df['contratado'] == 1]['FPD'].unique())    # OK Faz sentido ter apaenas valores [0. 1.] para contratados
print(df[df['contratado'] == 0]['FPD'].unique())    # OK Faz sentido ter apaenas valores [nan] para não contratados

print(df[df['contratado'] == 1]['target'].unique()) # OK Faz sentido ter apaenas valores [0. 1.] para contratados
print(df[df['contratado'] == 0]['target'].unique()) # OK Faz sentido ter apaenas valores [nan] para não contratados

🥚 **Easter Egg identificado: Viés de sobrevivência — target=0 imaturado em jan/25+**

**Evidência numérica:** 168.000 registros (jan/25–dez/25) tinham `target = 0` na base bruta, sem nenhum caso de `target = 1` — impossível em qualquer carteira real.

- **O que é (Easter Eggs #4 e #5):**
  - **Easter Egg #4 — Janela de target:** O target de 12 meses só pode ser considerado válido para contratos firmados até dez/24. Contratos de jan/25 em diante ainda não completaram 12 meses de observação (referência: base analisada em março/2026).
  - **Easter Egg #5 — Viés de sobrevivência:** Na base bruta, todos os 168.000 registros de jan/25–dez/25 tinham `target = 0`. Isso não significa que esses clientes são adimplentes — significa que ainda não tivemos tempo suficiente para observar se vão inadimplir. Incluir esses zeros falsificaria a taxa de inadimplência do modelo como se fosse 0%.
- **Como foi detectado:** `df[df['janela'] == 'target_vazio']['target'].unique()` retornou `[0, nan]` — havia zeros que deveriam ser `NaN`. Após correção: retorna apenas `[nan]` ✅.
- **Implicação para o modelo:** Se os 168.000 registros imaturados fossem mantidos como `target = 0`:
  - Taxa de inadimplência geral cairia artificialmente de ~13% para ~8%
  - KS e Gini calculados sobre essa base seriam inflados artificialmente
  - O modelo pareceria melhor do que é
- **Recomendação de tratamento:** Substituir `target = 0` por `NaN` para o período `target_vazio` (conforme feito nesta célula) e criar a flag `target_disponivel = (contratado == 1) & (ano_mes <= dez/24)` para filtrar análises de discriminação e previsibilidade.
- **Narrativa para apresentação:** "168 mil contratos recentes tinham inadimplência zerada na base — não porque os clientes são ótimos pagadores, mas porque ainda não passou tempo suficiente para saber. Usar esse zero como verdade inflaria artificialmente as métricas de performance do modelo."

> Aqui pode fazer sentido para o negócio manter target 0 neste período, pois os clientes contrataram e ainda não inadimpliram, 
> ou seja, estão no início do ciclo de crédito e ainda não tiveram tempo suficiente para se tornarem inadimplentes.
> 
> Mas manter target 0 na análise pode distorcer informações ou métricas de performance, pois a informação está incompleta. 


In [ ]:
df['vl_bem'] = df['vl_bem'].replace('25a70', '(25000, 70000]')

In [ ]:
# Verificar os valores únicos da coluna 'target' para a janela 'target_vazio'
print(df[df['janela'] == 'target_vazio']['target'].unique()) # NOT OK

# Vamos substituir os valores 0 da coluna 'target' para NaN, pois a janela 'target_vazio' representa clientes que não tem target completamente maduro.
idx = df[(df['janela'] == 'target_vazio') & (df['target'] == 0)].index
df.loc[df.index.isin(idx), 'target'] = np.nan
print(df[df['janela'] == 'target_vazio']['target'].unique()) # OK
df[df.index.isin(idx)]
                                                              

In [ ]:
# Coluna para tipo de cliente (adimplente, inadimplente, não contratado)
# Definir condições
condicoes = [
    (df['contratado'] == 0),                        # Não Contratado (Ponto Cego)
    (df['contratado'] == 1) & (df['target'] == 0),  # Adimplente
    (df['contratado'] == 1) & (df['target'] == 1)   # Inadimplente
]

# Definir valores correspondentes
valores = [
    'nao_contratado',
    'adimplente',
    'inadimplente',
]

# Criar a coluna tipo
df['tipo'] = np.select(condicoes, valores, default='Desconhecido')
df['tipo'].unique()

# Imprimir o total de registros em cada grupo
print("---" ) 
print("QUANTIDADE DE REGISTROS POR TIPO")
total_base = len(df)
total_contratados = len(df[df['tipo'].isin(['adimplente','inadimplente'])])

# Imprimir o total de registros em cada grupo
print("---" ) 
print("QUANTIDADE DE REGISTROS POR TIPO")
print("  1. Não Contratados : ", f"{len(df[df['tipo'] == 'nao_contratado']):,}", f"| {len(df[df['tipo'] == 'nao_contratado'])/total_base:.2%} da base")
print("  2. Contratados     : ", f"{len(df[df['tipo'].isin(['adimplente','inadimplente'])]):,}", f"| {len(df[df['tipo'].isin(['adimplente','inadimplente'])])/total_base:.2%} da base")
print("     - Adimplente    : ", f"{len(df[df['tipo'] == 'adimplente']):,}", f"| {len(df[df['tipo'] == 'adimplente'])/total_contratados:.2%} dos Contratados")
print("     - Inadimplente  : ", f"{len(df[df['tipo'] == 'inadimplente']):,} ", f"| {len(df[df['tipo'] == 'inadimplente'])/total_contratados:.2%} dos Contratados")

# Imprimir o período de análise por grupo
print("\n" "---" ) 
print("PERÍODO DE ANÁLISE POR TIPO")
print(f"  1. Não Contratados :  {df[df['tipo'] == 'nao_contratado']['ano_mes'].min()} a {df[df['tipo'] == 'nao_contratado']['ano_mes'].max()}")
print(f"  2. Contratados     :  {df[df['tipo'].isin(['adimplente','inadimplente'])]['ano_mes'].min()} a {df[df['tipo'].isin(['adimplente','inadimplente'])]['ano_mes'].max()}")
print(f"     - Adimplente    :  {df[df['tipo'] == 'adimplente']['ano_mes'].min()} a {df[df['tipo'] == 'adimplente']['ano_mes'].max()}")
print(f"     - Inadimplente  :  {df[df['tipo'] == 'inadimplente']['ano_mes'].min()} a {df[df['tipo'] == 'inadimplente']['ano_mes'].max()}")


In [ ]:
df['tipo'].unique()

# 1. EDA

EASTER EGG's:
1. Erro de base: vl_bem = 25a70;
2. Erro de base: Tem 347 id's que repetem, sendo que a repetição máxima é de 2 vezes e claramente não se tratam da mesma pessoa e nem do mesmo contrato;
3. score: Os scores de 1 a 4 foram descontinuados, então não devem fazer parte da análise:
   - De monitoramento de perfil;
4. qt_restr: 

## 1.0. Análise por feature

PERÍODO                | DESCRIÇÃO            | Observação         
--                     | --                   | --
['202203' - '202303')  | Janela de modelagem  | 
['202303' - '202403')  | Janela de observação | ['202303']: Marcador, <br>['202309']: Efeito Modelo Teste
['202403' - '202405')  | Limbo                | 
['202405']             | Implantado           | 
['202405' - 202512]    | Em produção          | ['202410', '202501'): Efeito Modelo Produção, <br>['202501', '202512'): Não tem taget = 1, <br>['202503']: Marcador

## 1.0.1. Variáveis Categóricas

FEATURE       | Mudou? | OBSERVAÇÃO
--            | --     | --
score         |        | **
FPD           |        | Em 2025 não tem inadimplentes, mas tem FPD's
Profissao     | não    | 
Regiao        |        | 
idade_veiculo |        | 
vl_bem        |        | 
qt_restr      |        | 
marca         |        | 
estado_civil  |        | 
entrada       |        | 
prazo         |        | 

## 1.0.2. Variáveis Numéricas

FEATURE   | OBSERVAÇÃO
--        | --
idade     | 
pmt       | 
renda     | 


## 0.1. Filtro e seleção

In [ ]:
# ============================================================
# CONFIGURAÇÃO GERAL — altere aqui, não nas células de análise
# ============================================================
CAMINHO_DADOS         = "../data/2_envio_case_base_Total_2026.csv"

# Janelas Temporais
DT_MODELAGEM_INICIO   = '2022-03-01'
DT_MODELAGEM_FIM      = '2023-02-01'
DT_OBSERVACAO_INICIO  = '2023-03-01'
DT_LIMBO_INICIO       = '2024-03-01'
DT_PRODUCAO_INICIO    = '2024-05-01'
DT_VAZIO_INICIO       = '2025-01-01'
DT_OBSERVACAO_PSI     = '2024-12-01'

SCORE_MINIMO          = 4
SIGMA_CONTROLE        = 3

COLS_NUMERICAS        = ['idade', 'pmt', 'renda']
COLS_CATEGORICAS_EXTRA = ['FPD', 'entrada', 'prazo', 'score']

COR_MODELAGEM  = 'lightgreen'
COR_OBSERVACAO = 'lightblue'
COR_LIMBO      = 'dimgray'
COR_PRODUCAO   = 'lightcoral'
COR_VAZIO      = 'lightgray'
COR_SPIKE      = 'orange'

In [ ]:
# Separar os dados em contratados e não contratados
df_nao_contratado = df[(df['tipo'] == 'nao_contratado') & 
                       (df['score'] > 4)].copy()  
                       
df_contratado     = df[(df['tipo'].isin(['adimplente','inadimplente'])) & 
                       (df['score'] > 4)].copy() 

# Separar os dados em contratados inadimplentes e adimplentes
df_adimplente     = df[(df['tipo'] == 'adimplente') &
                       (~df['janela'].isin(['target_vazio']))& 
                       (df['score'] > 4)].copy()     
                       
df_inadimplente   = df[(df['tipo'] == 'inadimplente') &
                       (~df['janela'].isin(['target_vazio']))& 
                       (df['score'] > 4)].copy() 

🥚 **Easter Egg identificado: Scores 1–4 descontinuados distorcem análise de estabilidade**

- **O que é:** Os scores de 1 a 4 existem na base histórica mas foram descontinuados como política de crédito — a empresa parou de aprovar/rejeitar contratos com esses scores em determinado momento. Portanto, eles não refletem o comportamento atual do modelo em operação.
- **Como foi detectado:** A análise de datas de início e fim por score (`df.groupby(['score'])['ano_mes'].agg(['min','max'])`) mostra que scores 1–4 têm data `max` diferente dos demais, indicando que deixaram de ser atribuídos antes do fim do período.
- **Implicação para o modelo:** Incluir scores 1–4 no cálculo do PSI cria uma falsa instabilidade: a concentração nessas faixas diminui ao longo do tempo simplesmente porque deixaram de ser usadas, não porque a população mudou de perfil. O PSI ficaria artificialmente elevado.
- **Recomendação de tratamento:** Filtrar `score > 4` em todas as análises de estabilidade (PSI, carta de controle, CSI). Os scores 1–4 podem ser mantidos para análise histórica isolada com nota explicativa.
- **Narrativa para apresentação:** "Parte dos scores do modelo foi desativada como política — mantê-los na análise de estabilidade criaria um alarme falso de que o perfil da população mudou drasticamente, quando na verdade foi apenas uma mudança de regra de negócio."

In [ ]:
df_nao_contratado['score'].unique()

In [ ]:
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
cat_cols = cat_cols + ['FPD', 'entrada', 'prazo', 'score']

cat_cols = sorted(cat_cols)
num_cols = sorted(['idade', 'pmt', 'renda'])

print("Colunas categóricas : ", cat_cols)
print("Colunas numéricas   : ", num_cols)

🥚 **Easter Egg identificado: `qt_restr` e `idade_veiculo` tipadas como string**

- **O que é:** O output acima mostra que as colunas `qt_restr` (quantidade de restrições no CPF) e `idade_veiculo` aparecem em `cat_cols` — a lista de variáveis categóricas — quando deveriam ser numéricas. Isso ocorre porque ambas foram importadas como `object` (string) pelo pandas, provavelmente por conterem valores não numéricos em algum registro (por exemplo, valores como `'None'`, `''` ou formatos não padrão).
- **Como foi detectado:** `df.select_dtypes(exclude=[np.number])` retornou essas colunas na lista de categóricas, confirmando que o tipo inferido não é numérico. A inclusão em `cat_cols` faz com que sejam tratadas como categorias na análise visual, o que mascara sua natureza ordinal.
- **Implicação para o modelo:** Qualquer operação aritmética sobre essas colunas (média, desvio padrão, histograma, CSI numérico) falhará silenciosamente ou produzirá resultados incorretos. O cálculo de WoE/IV para `qt_restr` como string cria categorias artificiais baseadas no texto, não nos valores numéricos.
- **Recomendação de tratamento:** Converter com `pd.to_numeric(df['qt_restr'], errors='coerce')` e `pd.to_numeric(df['idade_veiculo'], errors='coerce')`. O `errors='coerce'` transforma valores inválidos em NaN (ao invés de levantar exceção), que pode então ser tratado separadamente (Aguiar et al. recomendam criar uma categoria "Sem informação" nesses casos).
- **Narrativa para apresentação:** "Duas variáveis chegaram ao modelo como texto quando deveriam ser números — o sistema de origem estava gravando a quantidade de restrições e a idade do veículo como se fossem palavras, não quantidades. Isso faz cálculos falharem de forma silenciosa, sem mensagem de erro aparente."

## 1.1. Análise Descritiva

In [ ]:
df_contratado.describe().T

📊 **Leitura da métrica: Análise Descritiva da população contratada**

- **O que a métrica diz:** O `describe()` da base de contratados fornece o perfil estatístico da população que o modelo efetivamente decidiu aprovar — é a "foto" da carteira em monitoramento.
- **Pontos de atenção ao interpretar:**
  - `score`: Verificar se a distribuição central está compatível com o DEV. Shifts no score médio indicam mudança de política ou perfil (antecipa PSI).
  - `pmt` e `renda`: Avaliar razão prestação/renda. Deterioração nessa razão é sinal de risco crescente.
  - `idade`: Confirmar que o perfil etário não mudou substancialmente desde o desenvolvimento.
- **Implicação para o modelo em produção:** Se as estatísticas de produção diferem muito do DEV, o modelo pode estar sendo aplicado numa população diferente daquela para qual foi treinado — o que justifica redução de performance.
- **Recomendação para a equipe de ciência de dados:** Comparar estas estatísticas com as do período de modelagem (DEV: mar/22–fev/23). Diferenças acima de 1 desvio padrão em variáveis preditoras merecem análise de CSI (Characteristic Stability Index).
- **Narrativa para apresentação:** "Antes de interpretar qualquer métrica de performance, verificamos se o modelo está sendo aplicado para o mesmo tipo de cliente para o qual foi desenvolvido — pequenas diferenças no perfil médio já nos dão os primeiros sinais de alerta."

In [ ]:
# Quantidade de vezes que cada id aparece
# Ordenar por quantidade de ocorrências
id_counts = df['id'].value_counts()
id_counts_sorted = id_counts.sort_values(ascending=False)

# Mostrar apenas os ids que aparecem mais de 1 vez
id_counts_sorted[id_counts_sorted > 1]

🥚 **Easter Egg identificado: IDs duplicados na base**

- **O que é:** A base contém 347 IDs de clientes que aparecem mais de uma vez, com no máximo 2 ocorrências por ID. Ao inspecionar os registros duplicados, percebe-se que são claramente contratos distintos (datas diferentes, variáveis diferentes) vinculados ao mesmo identificador — não são duplicatas acidentais de uma mesma proposta.
- **Como foi detectado:** `df['id'].value_counts()` revelou 347 IDs com frequência > 1. Análise dos pares confirma que as demais colunas (data, score, etc.) diferem entre os registros do mesmo ID.
- **Implicação para o modelo:** Manter esses registros duplicados infla a contagem de propostas e, se o cliente inadimpliu em um dos contratos, o KS e o Gini são distorcidos (um mesmo cliente conta duas vezes com behaviors possivelmente distintos). O PSI também é afetado por excesso de peso nas faixas de score onde esse cliente se concentra.
- **Recomendação de tratamento:** Criar uma chave composta `id + data` como identificador único (Aguiar et al. recomendam validar granularidade da chave antes de qualquer análise). Para análises de discriminação, usar `drop_duplicates(subset='id', keep='first')` como solução temporária até a equipe de dados corrigir a origem.
- **Narrativa para apresentação:** "347 clientes aparecem duas vezes na base com contratos diferentes — isso significa que o modelo estava sendo avaliado com mais dados do que realmente existiam, potencialmente inflando suas métricas de performance."

## 1.2. Análise Visual

### 1.2.1. Variáveis Categóricas

- entrada
  - O valor de entrada 0 ter baixa inadimplência é estranho, mas faz sentido que % seja maior do que na base de adimplentes (15% > 10%) E ele está de acordo com a janela de modelagem;
- idade_veiculo
  - Não faz sentido que 

In [ ]:
cat_cols

In [ ]:
cols = cat_cols

for coluna in cols:
    if coluna in ['ano_mes', 'FPD', 'tipo', 'janela']:
        continue
    plot_distribuicao_por_grupos(df_contratado, df_adimplente, df_inadimplente, col=coluna)

📊 **Leitura da EDA: Análise Visual de Variáveis Categóricas ao longo do tempo**

- **O que a análise diz:** Os gráficos de distribuição ao longo do tempo para cada variável categórica comparam três grupos: contratados totais, adimplentes e inadimplentes. Essa segmentação permite identificar se determinada categoria concentra mais risco e se essa concentração mudou entre DEV e produção.
- **Comparação entre períodos:** Se a distribuição de uma variável muda entre a janela de modelagem (mar/22–fev/23) e a janela de produção (mai/24+), isso é um sinal de CSI elevado — a população que está chegando é diferente da que treinou o modelo.
- **Sinais críticos a observar:**
  - `score`: Presença de scores 1–4 no histórico mas ausência na produção (Easter Egg #3 confirmado visualmente).
  - `vl_bem`: Presença/ausência da categoria `'25a70'` (Easter Egg #1 confirmado visualmente).
  - `entrada`: Concentração no valor 0 com baixa inadimplência pode ser contraintuitivo — verificar se é efeito de seleção (clientes sem entrada tendem a ser mais qualificados neste produto).
  - `idade_veiculo`: Observar se houve mudança no mix de veículos novos vs usados ao longo do tempo.
- **Implicação para o modelo em produção:** Variáveis com distribuição muito diferente entre DEV e produção requerem CSI formal para quantificar o impacto na estabilidade do modelo.
- **Narrativa para apresentação:** "As distribuições das variáveis ao longo do tempo nos mostram quem são os clientes que chegaram em cada período — mudanças visíveis são o primeiro sinal de que o modelo pode estar operando fora de seu domínio de treinamento."

In [ ]:
# Para todos os scores
resultado = df[df['contratado'] == 1].groupby(['score'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável score:')
print(resultado)

# Para todos os Profissao
resultado = df[df['contratado'] == 1].groupby(['Profissao'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável Profissao:')
print(resultado)

# Para todos os Regiao
resultado = df[df['contratado'] == 1].groupby(['Regiao'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável Regiao:')
print(resultado)

# Para todas as marcas
resultado = df[df['contratado'] == 1].groupby(['marca'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável marca:')
print(resultado)

# Para todas as entrada
resultado = df[df['contratado'] == 1].groupby(['entrada'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável entrada:')
print(resultado)

# Para todas as estado_civil
resultado = df[df['contratado'] == 1].groupby(['estado_civil'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável estado_civil:')
print(resultado)

# Para todas as idade_veiculo
resultado = df[df['contratado'] == 1].groupby(['idade_veiculo'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável idade_veiculo:')
print(resultado)

# Para todos os qt_restr
resultado = df[df['contratado'] == 1].groupby(['qt_restr'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável qt_restr:')
print(resultado)

# Para todos os vl_bem
resultado = df[df['contratado'] == 1].groupby(['vl_bem'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável vl_bem:')
print(resultado)

# Para todos os prazos
resultado = df[df['contratado'] == 1].groupby(['prazo'])['ano_mes'].agg(['min', 'max'])
print("\n" "---" ) 
print('Datas de início e fim da variável prazo:')
print(resultado)

🥚 **Easter Egg identificado: Spike de aprovação +10 p.p. (out/24–mar/25) + Queda de volume -40% (abr/25+)**

**Evidência numérica:** Taxa de aprovação sobe de ~76% para ~85.7–86.0% em out/24, persiste por 6 meses, e retorna a ~76% em abr/25. Simultaneamente, o volume mensal de propostas cai de ~20.000 para ~12.000 em abr/25.

- **O que é (Easter Egg #9 — Spike):** A taxa de aprovação `%_media_contratado` sobe abruptamente de ~76% para ~85.7% em outubro/2024 e permanece elevada até março/2025 (6 meses consecutivos), retornando ao patamar histórico em abril/2025. Essa elevação de ~10 p.p. não foi gradual — foi uma descontinuidade abrupta.
- **O que é (Easter Egg #10 — Queda de volume):** A partir de abril/2025, o volume total de propostas cai de ~20.000/mês para ~12.000/mês — redução de **~40%** que se mantém estável em todos os meses subsequentes até dezembro/2025.
- **Como foram detectados:** Análise da tabela dinâmica `%_media_contratado` por `ano_mes` revela o spike; contagem de registros por `ano_mes` revela a queda de volume.
- **Implicação para o modelo — Spike:** Um aumento abrupto de 10 p.p. na taxa de aprovação indica que o ponto de corte do score foi alterado (regra de negócio), ou que a composição das propostas mudou drasticamente. O efeito: ~10% a mais de propostas aprovadas nos 6 meses do spike representam um cohort de maior risco que ainda não tem target maturado (contratos de out/24 terão target em out/25, que está fora do período analisado).
- **Implicação para o modelo — Queda de volume:** Com apenas 12.000 propostas/mês vs 20.000 histórico, as proporções por faixa de score calculadas no PSI são menos estáveis estatisticamente. O PSI calculado com volumes menores tende a ser mais volátil — o limite de 0.10 pode ser atingido por variação amostral, não por mudança real de perfil.
- **Recomendação de tratamento:** (1) Confirmar com o time de negócio o motivo do spike e da queda de volume; (2) normalizar volumes no PSI para comparações mais robustas; (3) documentar os períodos afetados no relatório de monitoramento.
- **Narrativa para apresentação:** "Em outubro de 2024, algo mudou na política de aprovação — 1 em cada 10 clientes que seria recusado passou a ser aprovado. Seis meses depois, o volume de novos contratos caiu 40%. Esses dois eventos simultâneos são sinais claros de mudanças operacionais que o modelo não estava preparado para absorver."

### 1.2.2. Variáveis Numéricas

In [ ]:
cols = num_cols

for coluna in cols:
    plot_estatisticas_por_tipo(df, coluna)

📊 **Leitura da EDA: Análise Visual de Variáveis Numéricas ao longo do tempo**

- **O que a análise diz:** Os gráficos de média, mediana e coeficiente de variação para `idade`, `pmt` (prestação) e `renda` ao longo do tempo revelam tendências nos perfis financeiros dos clientes — tanto contratados quanto segmentados por adimplência.
- **Pontos críticos a observar:**
  - **`renda`**: O boxplot por janela e tipo mostra se a renda média dos inadimplentes está convergindo para a dos adimplentes — o que indicaria deterioração do poder discriminatório da variável.
  - **`pmt`**: Prestações crescentes sem renda crescente elevam o comprometimento de renda, aumentando risco.
  - **`idade`**: Mudança no perfil etário pode refletir mudança no mix de produto (veículo novo vs usado, por exemplo) ou na política de aprovação.
- **Comparação entre períodos:** Diferenças entre DEV (mar/22–fev/23) e produção (mai/24+) nessas variáveis numéricas devem ser quantificadas via CSI para determinar se são estatisticamente relevantes.
- **Implicação para o modelo em produção:** Coeficiente de variação crescente indica que a população está se tornando mais heterogênea — o modelo foi treinado em uma população mais homogênea e pode ter dificuldade de discriminar em populações mais dispersas.
- **Narrativa para apresentação:** "A renda e o valor da prestação dos clientes mudaram ao longo do tempo — quando essa mudança é sistemática, ela explica por que um modelo treinado no passado começa a errar no presente."

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurar o estilo do gráfico
sns.set_style("whitegrid")
plt.figure(figsize=(12, 6))

# Criar o boxplot
sns.boxplot(data=df, x='janela', y='renda', hue='tipo')

plt.title('Distribuição de Renda por Tipo e Janela')
plt.xlabel('Janela')
plt.ylabel('Renda')
plt.legend(title='Tipo')
plt.xticks(rotation=45)  # Se necessário, rotacionar os labels
plt.tight_layout()
plt.show()

In [ ]:
# Para todas as idades
resultado = df[df['contratado'] == 1].groupby(['idade'])['ano_mes'].agg(['min', 'max'])
print("\n" + "---")
print('Datas de início e fim para cada idade:')
resultado[resultado['max'] != '2025-12-01']  # Idades que só aparecem a partir da janela de observação

In [ ]:
# Para todas as pmt = Valor da prestação
resultado = df[df['contratado'] == 1].groupby(['pmt'])['ano_mes'].agg(['min', 'max'])
print("\n" + "---")
print('Datas de início e fim para cada pmt:')
resultado[resultado['max'] != '2025-12-01'].sort_values('max')  # Pmts que só aparecem a partir da janela de observação

In [ ]:
# Para todas as renda = Valor da renda
resultado = df[df['contratado'] == 1].groupby(['renda'])['ano_mes'].agg(['min', 'max'])
print("\n" + "---")
print('Datas de início e fim para cada renda:')
resultado.sort_values('max')  # Rendas que só aparecem a partir da janela de observação

# 2. Métricas de Monitoramento

1. **Estabilidade**:
   - Avalia se o perfil da população mudou.
2. **Discriminação**
   - Avalia se o modelo **ainda consegue separar corretamente os bons dos maus pagadores**;
   - Mede a capacidade do modelo em ordenar os clientes segundo o critério para o qual foi desenvolvido;
   - Uma queda no poder discriminatório pode indicar que as relações aprendidas pelo modelo durante o desenvolvimento não são mais válidas, comprometendo todas as decisões baseadas nele.
3. **Previsibilidade**
   - sadf


## 2.1. Estabilidade

In [ ]:
# Selecionar as informações para análise
#filtro = ((df['tipo']  == 'adimplente') &
#          (df['score'] > 4))
df_analise_estabilidade = df_inadimplente.copy()

# Selecionar as datas para análise de PSI 
dt_esperada_inicio = '2022-03-01' # Início da Janela de Modelagem
dt_esperada_fim    = '2023-02-01' # Fim da Janela de Modelagem
dt_observacao      = '2024-12-01' # Data da análise

📊 **Configuração da análise de estabilidade — contexto importante**

- **O que foi feito:** A análise de estabilidade usa `df_inadimplente` como base, com período esperado DEV (mar/22–fev/23) e data de observação dez/24.
- **Atenção metodológica:** Para o PSI do score, a referência deve ser toda a população contratada no DEV (não apenas inadimplentes). Usar apenas inadimplentes como base de comparação cria uma visão parcial do PSI — o PSI mede a estabilidade da *distribuição do score na população total*, não apenas nos maus pagadores.
- **O que isso significa:** Se esta célula usa `df_inadimplente`, o PSI calculado a seguir mede a estabilidade do score *entre os inadimplentes* — útil como análise complementar, mas diferente do PSI padrão que compara toda a população entre DEV e produção.
- **Recomendação:** Para o PSI oficial (Aguiar et al.), recalcular usando `df_contratado` (toda a base contratada, com `score > 4`) como população de análise.

In [ ]:
col = 'score'

# Tabela dinâmica de frequência relativa por ano_mes e score
tabela_frequencia = pd.crosstab(df_analise_estabilidade['ano_mes'], df_analise_estabilidade[col], normalize='index')
tabela_frequencia.head(2)

### 2.1.1. PSI ou IEP (Índice de Estabilidade do Populacional)

In [ ]:
# Esperado = a média dos valores do período entre 2022-03 e 2023-03 (Janela de Modelagem) para cada score.
esperado = tabela_frequencia.loc[dt_esperada_inicio:dt_esperada_fim].mean()

# Observado = os valores de uma data específica (ex: 2024-03) para cada score.
observado = tabela_frequencia.loc[[dt_observacao]].iloc[0]
observado

# Criar DataFrame com diferença
df_psi = pd.DataFrame({
    'Esperado': esperado,
    'Observado': observado,
    'PSI': (observado - esperado) * np.log(observado / esperado) # PSI = (%Obs - %Esp) * ln(%Obs / %Esp)
})

PSI = df_psi['PSI'].sum()
print(f"PSI Total: {PSI:.4f}")


In [ ]:
df_psi.T

📊 **Leitura da métrica: PSI do Score (DEV mar/22–fev/23 vs dez/24)**

**Resultado obtido: PSI Total = 0.0019** — classificação: **Estável ✅**

- **O que a métrica diz:** O PSI (Population Stability Index, também chamado de IEP — Índice de Estabilidade Populacional) mede o quanto a distribuição do score mudou entre o período de referência (modelagem) e o período atual (produção). PSI = Σ (Observado% − Esperado%) × ln(Observado% / Esperado%).
- **Detalhe por faixa de score:**
  | Score | Esperado (DEV) | Observado (dez/24) | PSI parcial |
  |-------|---------------|-------------------|-------------|
  | 5 | 27.7% | 26.9% | 0.000240 |
  | 6 | 28.6% | 28.2% | 0.000053 |
  | 7 | 27.0% | 27.2% | 0.000018 |
  | 8 | 12.4% | 13.5% | 0.001070 |
  | 9 | 3.9% | 3.9% | 0.000008 |
  | 10 | 0.47% | 0.33% | 0.000514 |
- **Thresholds de referência (Aguiar et al.):** PSI < 0.10 → Estável ✅ | 0.10–0.25 → Atenção ⚠️ | > 0.25 → Instável ❌
- **Interpretação:** Com PSI = 0.0019, a distribuição do score entre os inadimplentes em dezembro/2024 é praticamente idêntica à distribuição no período de modelagem. Nenhuma faixa de score concentra instabilidade relevante — o maior contribuinte individual é o score 8 com PSI parcial de apenas 0.001.
- **Atenção metodológica importante:** Este PSI foi calculado sobre a base de **inadimplentes** (`df_inadimplente`). O PSI padrão da literatura (Aguiar et al.) é calculado sobre toda a **população contratada**. Embora o resultado aqui seja estável, o PSI oficial deve ser recalculado com `df_contratado` para completar o relatório de monitoramento.
- **Implicação para o modelo em produção:** O perfil de score dos inadimplentes não mudou — o modelo está vendo o mesmo tipo de cliente ruim. Isso é um sinal positivo para a previsibilidade.
- **Narrativa para apresentação:** "O PSI de 0.0019 confirma que o perfil de risco dos clientes que o modelo reconhece como maus pagadores permaneceu praticamente inalterado desde o treinamento — o modelo está operando sobre o mesmo tipo de população."

### 2.1.2. Carta de Controle

#### 2.1.2.1. Controle do observado (dentro dos limites esperados por score)

In [ ]:
# Carta de controle do observado
esperado_std = tabela_frequencia.loc[dt_esperada_inicio:dt_esperada_fim].std()

# Criar DataFrame com diferença
df_controle = pd.DataFrame({
    'Esperado': esperado,
    'Observado': observado,        
    'LS': (esperado + 2 * esperado_std), # Limite Superior = Esperado + 2 * Desvio Padrão
    'LI': (esperado - 2 * esperado_std),  # Limite Inferior = Esperado - 2 * Desvio Padrão
    'Cruzou?': (observado > (esperado + 2 * esperado_std)) | (observado < (esperado - 2 * esperado_std))
})

df_controle.T

In [ ]:
# Plot da frequência observada por score ao longo do tempo, comparando com LS/LI de cada score
scores = df_controle.index.to_list()
n_scores = len(scores)
n_colunas = 2
n_linhas = (n_scores + n_colunas - 1) // n_colunas  # Arredondar para cima

fig, axes = plt.subplots(n_linhas, n_colunas, figsize=(16, 3 * n_linhas))
axes = axes.flatten()

for i, score in enumerate(scores):
    ax = axes[i]
    serie = tabela_frequencia[score]
    
    ax.plot(serie.index, serie.values, label='Observado', marker='o', linewidth=1.5)
    ax.axhline(df_controle.loc[score, 'LS'], color='red', linestyle='--', label='LS')
    ax.axhline(df_controle.loc[score, 'LI'], color='green', linestyle='--', label='LI')
    
    ax.axvspan('2022-03-01', '2023-03-01', color='lightgreen', alpha=0.3)                    # Janela de Modelagem
    ax.axvspan('2023-03-01', '2024-03-01', color='lightblue', alpha=0.3)                     # Janela de Observação
    ax.axvspan('2024-05-01', tabela_frequencia.index.max(), color='lightpink', alpha=0.3)    # Em Produção
    ax.axvspan('2024-03-01', '2024-05-01', color='black', alpha=0.4)                         # Limbo
    ax.axvspan('2023-03-01', '2023-03-01', color='black', alpha=0.4)                         # Marcador
    ax.axvspan('2023-09-01', '2023-09-01', color='black', alpha=0.4)                         # Efeito Modelo Teste
    ax.axvspan('2025-03-01', '2025-03-01', color='black', alpha=0.4)                         # Marcador
    ax.axvspan('2024-10-01', '2025-01-01', color='orange', alpha=0.4)                        # Efeito Modelo Produção
    ax.axvspan('2025-01-01', tabela_frequencia.index.max(), color='lightgray', alpha=0.4)    # Não tem target 1
    
    ax.set_title(f'Score {score}')
    ax.set_xlabel('Ano-Mês')
    if i % n_colunas == 0:
        ax.set_ylabel('Freq. relativa')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

# Esconder subplots vazios (se houver)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

📊 **Leitura da métrica: Carta de Controle por Score Individual (Pilar 1 — Estabilidade)**

**Resultado: Nenhuma faixa de score cruzou os limites de controle em dezembro/2024** — classificação: **Estável ✅**

- **Limites de controle calculados (DEV mar/22–fev/23 como referência):**
  | Score | Esperado | Observado (dez/24) | Limite Inf. | Limite Sup. | Cruzou? |
  |-------|----------|-------------------|-------------|-------------|---------|
  | 5 | 27.7% | 26.9% | 25.7% | 29.6% | Não ✅ |
  | 6 | 28.6% | 28.2% | 24.5% | 32.6% | Não ✅ |
  | 7 | 27.0% | 27.2% | 23.8% | 30.2% | Não ✅ |
  | 8 | 12.4% | 13.5% | 10.1% | 14.6% | Não ✅ |
  | 9 | 3.9% | 3.9% | 2.4% | 5.5% | Não ✅ |
  | 10 | 0.47% | 0.33% | 0.10% | 0.85% | Não ✅ |
- **Interpretação dos gráficos:** Os 6 gráficos individuais por score mostram a evolução mensal da frequência de cada nota ao longo de toda a série (mar/22–dez/24). As linhas tracejadas em vermelho (LS) e verde (LI) delimitam o corredor de normalidade. Visualmente, a série fica dentro dos limites em todos os meses analisados para dezembro/2024.
- **Atenção:** Os gráficos foram calculados sobre `df_inadimplente` — a estabilidade observada reflete a constância no perfil de score dos maus pagadores, não da população total. O comportamento da série temporal deve ser monitorado mensalmente para detectar tendências de deslocamento.
- **Implicação para o modelo em produção:** O perfil de score dos inadimplentes está dentro do padrão histórico. Não há faixa de score com volume anômalo que justifique investigação imediata de CSI.
- **Recomendação para a equipe de ciência de dados:** Monitorar especialmente o score 8, que apresentou o maior PSI parcial (0.001) e o score 10, que mostrou frequência observada ligeiramente abaixo do esperado — ambos dentro dos limites, mas merecem atenção continuada.
- **Narrativa para apresentação:** "Mês a mês, a proporção de clientes inadimplentes em cada faixa de score ficou dentro do corredor esperado — o modelo está reconhecendo os mesmos padrões de risco que aprendeu no treinamento."

#### 2.1.2.2. Controle da variação média ponderada por score

In [ ]:
# Carta de Controle

# Média ponderada do score por mês (soma produto = frequência * score)
scores = tabela_frequencia.columns.to_list()
media_ponderada_score_por_mes = tabela_frequencia.dot(scores)

# Desvio padrão do período da janela de modelagem (2022-03 a 2023-02)
std_ponderado_esperado = media_ponderada_score_por_mes.loc[dt_esperada_inicio:dt_esperada_fim].std() 
mean_ponderado_esperado = media_ponderada_score_por_mes.loc[dt_esperada_inicio:dt_esperada_fim].mean()

# Adicionar limites de controle
limite_superior = mean_ponderado_esperado + 2 * std_ponderado_esperado # Limite Superior
limite_inferior = mean_ponderado_esperado - 2 * std_ponderado_esperado # Limite Inferior

In [ ]:
# Plotar a variação da média ponderada do score ao longo do tempo
plt.figure(figsize=(12, 6))
plt.plot(media_ponderada_score_por_mes.index, media_ponderada_score_por_mes.values, label='Média Ponderada do Score', color='blue')

# Adicionar linhas para os limites superior e inferior
plt.axhline(y=limite_superior, color='red', linestyle='--', label=f'Limite Superior ({limite_superior:.4f})')
plt.axhline(y=limite_inferior, color='green', linestyle='--', label=f'Limite Inferior ({limite_inferior:.4f})')

# Configurações do gráfico
plt.title('Variação da Média Ponderada do Score ao Longo do Tempo')
plt.xlabel('Ano-Mês')
plt.ylabel('Média Ponderada do Score')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

📊 **Leitura da métrica: Carta de Controle da Média Ponderada do Score (Pilar 1 — Estabilidade)**

**Resultado: Média DEV = 6.38 | LS = 6.46 | LI = 6.30 | Produção (mai–set/24): ABAIXO do LI ⚠️**

- **Valores calculados:**
  - Média ponderada DEV (mar/22–fev/23): **6.3777**
  - Limite Superior (LS = média + 2σ): **6.4571**
  - Limite Inferior (LI = média − 2σ): **6.2983**
- **Evolução crítica por período:**
  | Período | Média ponderada | Status |
  |---------|----------------|--------|
  | DEV (mar/22–fev/23) | ~6.38 | Referência ✅ |
  | OOT (mar/23–fev/24) | ~6.40–6.46 | Dentro dos limites ✅ |
  | Produção mai/24–set/24 | **6.26–6.31** | **Abaixo do LI ⚠️** |
  | Produção out/24–dez/24 | ~6.39–6.41 | Recuperou — dentro dos limites ✅ |
- **Interpretação:** Entre maio e setembro de 2024, a média ponderada do score dos inadimplentes caiu abaixo do limite inferior (6.2983), com mínimo de **6.264 em agosto/2024**. Isso significa que os inadimplentes do período de produção tinham, em média, scores mais baixos (maior risco percebido pelo modelo) do que os inadimplentes do DEV. A recuperação a partir de outubro/2024 sugere que o efeito foi temporário.
- **Implicação para o modelo em produção:** A queda temporária da média abaixo do LI indica que o modelo estava atribuindo scores mais baixos (piores) a clientes que depois inadimpliam — o que é, paradoxalmente, uma demonstração de discriminação ativa do modelo. Monitorar se essa tendência de queda retorna em 2025.
- **Recomendação para a equipe de ciência de dados:** Investigar a cohort de mai–set/24 com análise de CSI para entender se houve mudança de perfil das variáveis explicativas nesse período. A recuperação em out/24 pode estar relacionada ao spike de aprovação identificado (Easter Egg #9).
- **Narrativa para apresentação:** "A média do score dos inadimplentes caiu em 2024 — o modelo estava identificando corretamente clientes de maior risco. A recuperação posterior coincide com o período de aumento abrupto na taxa de aprovação, que merece investigação separada."

## 2.2. Discriminação (Previsão)

In [ ]:
# Selecionar as informações para análise
# Selecionar as informações para análise
filtro = ((df['tipo']  == 'adimplente') &
          (df['score'] > 4))
df_analise_estabilidade = df[filtro].copy()

# Selecionar as datas para análise de PSI 
dt_esperada_inicio = '2022-03-01' # Início da Janela de Modelagem
dt_esperada_fim    = '2023-02-01' # Fim da Janela de Modelagem
dt_observacao      = '2024-12-01' # Data da análise

In [ ]:
# Criar tabela dinâmica com % média de contratado == 1 e % média de target == 1 (entre os contratados)
tabela_contratado = df.groupby('ano_mes')['contratado'].mean().reset_index()
tabela_contratado.columns = ['ano_mes', '%_media_contratado']

tabela_target = df[df['contratado'] == 1].groupby('ano_mes')['target'].mean().reset_index()
tabela_target.columns = ['ano_mes', '%_media_inadimplente']

# Adicionar % média de FPD por ano_mes
tabela_fpd = df.groupby('ano_mes')['FPD'].mean().reset_index()
tabela_fpd.columns = ['ano_mes', '%_media_fpd']

# Mesclar as tabelas
tabela_dinamica = pd.merge(tabela_contratado, tabela_target, on='ano_mes', how='left')
tabela_dinamica = pd.merge(tabela_dinamica, tabela_fpd, on='ano_mes', how='left')

# Exibir a tabela
tabela_dinamica.tail(12)

In [ ]:
print('Previsão de Aprovação e Inadimplência):')
print('  % Média de Aprovação     : {:.2f}% (até 2025-12)'.format(tabela_dinamica['%_media_contratado'].mean() * 100))
# Calcular a média de % de inadimplência entre os contratados apenas até 2025-01 (antes do período sem target 1)
media_inadimplencia = tabela_dinamica[tabela_dinamica['ano_mes'] < '2025-01-01']['%_media_inadimplente'].mean()
print('  % Média de Inadimplência : {:.2f}% (até 2024-12)'.format(media_inadimplencia * 100))

📊 **Leitura da métrica: Taxa de Aprovação e Inadimplência ao longo do tempo (Pilar 2 — contexto)**

**Resultado: Taxa média de aprovação = 77.21% | Taxa média de inadimplência = 13.02% (até dez/24)**

- **O que a métrica diz:** A tabela consolidada mostra, mês a mês, a taxa de aprovação (`%_media_contratado`) e a taxa de inadimplência entre os aprovados (`%_media_inadimplente`).
- **Anomalia identificada — Easter Egg #9 e #10:**
  | Período | Taxa aprovação | Observação |
  |---------|---------------|------------|
  | DEV + OOT (mar/22–fev/24) | ~75–76% | Padrão estável ✅ |
  | Limbo (mar/24–abr/24) | ~75% | Excluído da análise ⚠️ |
  | Produção mai/24–set/24 | ~76–77% | Normal ✅ |
  | **out/24–mar/25** | **~85.7–86.0%** | **Spike de +10 p.p. ❌** |
  | abr/25–dez/25 | ~76% | Retorna ao normal ✅ |
- **Anomalia no volume mensal:** A partir de abril/2025, o volume total de propostas cai de ~20.000/mês para ~12.000/mês — redução de **40%** que persiste até dezembro/2025 (Easter Egg #10). Isso invalida o uso do PSI no período mai–dez/25, pois volumes muito diferentes distorcem as proporções.
- **Implicação para inadimplência futura:** O spike de aprovação de out/24–mar/25 (85.7% vs média histórica de 76%) indica que ~10% a mais de propostas foram aprovadas em 6 meses — ou seja, clientes que normalmente seriam recusados foram aprovados. Esses contratos ainda não têm target maturado (jan/25+), mas quando maturarem é esperado que a inadimplência nesse cohort seja maior.
- **FPD como sinal antecipado:** A taxa de FPD em jan–mar/25 (~4.9%) está acima da média histórica (~3.5–4.0%), o que é o único indicador disponível nesse período. Usar com cautela (Easter Egg #8).
- **Narrativa para apresentação:** "Entre outubro/2024 e março/2025, quase 1 em cada 6 propostas normalmente recusadas foi aprovada — seis meses de crédito mais frouxo. Quando esses contratos completarem 12 meses, esperamos ver uma elevação na inadimplência que este modelo não terá conseguido prever adequadamente."

### 2.2.1. KS

#### 2.2.1.1. Janela de Modelagem (OOT)

In [ ]:
#DT_MODELAGEM_INICIO   = '2022-03-01' # Treino
#DT_MODELAGEM_FIM      = '2023-02-01' # Treino

DT_MODELAGEM_INICIO   = '2023-03-01' # OOT
DT_MODELAGEM_FIM      = '2024-02-01' # OOT
SCORE_MINIMO          = 4

filtro_ks = ((df['ano_mes'] >= DT_MODELAGEM_INICIO) &
             (df['ano_mes'] <= DT_MODELAGEM_FIM) &
             (df['score'] > SCORE_MINIMO))
y_target = df[filtro_ks]['target'].values
y_score  = df[filtro_ks]['score'].values
ks, p_value = calcular_ks(y_target, y_score)
print(f"KS (Janela de Modelagem OOT): {ks:.4f} | p-valor: {p_value:.4e}")

In [ ]:
# Curva KS — padrão de mercado em crédito
# Mostra distribuição acumulada de bons e maus pagadores por faixa de score
df_modelagem_ks = df[
    (df['ano_mes'] >= DT_MODELAGEM_INICIO) &
    (df['ano_mes'] <= DT_MODELAGEM_FIM) &
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1)
].copy()

scores_range = sorted(df_modelagem_ks['score'].unique())
total_bons = (df_modelagem_ks['target'] == 0).sum()
total_maus = (df_modelagem_ks['target'] == 1).sum()

cdf_bons = [((df_modelagem_ks['target'] == 0) & (df_modelagem_ks['score'] <= s)).sum() / total_bons
            for s in scores_range]
cdf_maus = [((df_modelagem_ks['target'] == 1) & (df_modelagem_ks['score'] <= s)).sum() / total_maus
            for s in scores_range]
diffs    = [abs(b - m) for b, m in zip(cdf_bons, cdf_maus)]
ks_idx   = diffs.index(max(diffs))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(scores_range, cdf_bons, 'b-o', linewidth=2.5, markersize=7,
        label=f'Bons Pagadores (Adimplentes) — n={total_bons:,}')
ax.plot(scores_range, cdf_maus, 'r-o', linewidth=2.5, markersize=7,
        label=f'Maus Pagadores (Inadimplentes) — n={total_maus:,}')

ax.vlines(scores_range[ks_idx],
          min(cdf_bons[ks_idx], cdf_maus[ks_idx]),
          max(cdf_bons[ks_idx], cdf_maus[ks_idx]),
          color='black', linestyle='--', linewidth=2,
          label=f'KS = {max(diffs):.4f} (score {scores_range[ks_idx]})')

mid_y = (cdf_bons[ks_idx] + cdf_maus[ks_idx]) / 2
ax.annotate(f'  KS = {max(diffs):.4f}',
            xy=(scores_range[ks_idx], mid_y),
            fontsize=13, fontweight='bold', color='black')

ax.fill_between(scores_range, cdf_bons, cdf_maus, alpha=0.08, color='purple')

ax.set_xlabel('Score do Modelo', fontsize=12)
ax.set_ylabel('Frequência Acumulada', fontsize=12)
ax.set_title(f'Curva KS — Janela de Modelagem ({DT_MODELAGEM_INICIO[:7]} a {DT_MODELAGEM_FIM[:7]})\n'
             f'Scores ativos (>{SCORE_MINIMO}) | Apenas contratados', fontsize=13)
ax.set_xticks(scores_range)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

#### 2.2.1.2. Janela de Produção (Target Maduro) — Período de contratados com target 1 (Inadimplentes)

In [ ]:
DT_MODELAGEM_INICIO   = '2024-05-01' # Produção (Target Maduro)
DT_MODELAGEM_FIM      = '2024-12-01' # Produção (Target Maduro)
SCORE_MINIMO          = 4

filtro_ks = ((df['ano_mes'] >= DT_MODELAGEM_INICIO) &
             (df['ano_mes'] <= DT_MODELAGEM_FIM) &
             (df['score'] > SCORE_MINIMO))
y_target = df[filtro_ks]['target'].values
y_score  = df[filtro_ks]['score'].values
ks, p_value = calcular_ks(y_target, y_score)
print(f"KS (Janela de Produçao - Target Maduro): {ks:.4f} | p-valor: {p_value:.4e}")

In [ ]:
# Curva KS — padrão de mercado em crédito
# Mostra distribuição acumulada de bons e maus pagadores por faixa de score
df_modelagem_ks = df[
    (df['ano_mes'] >= DT_MODELAGEM_INICIO) &
    (df['ano_mes'] <= DT_MODELAGEM_FIM) &
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1)
].copy()

scores_range = sorted(df_modelagem_ks['score'].unique())
total_bons = (df_modelagem_ks['target'] == 0).sum()
total_maus = (df_modelagem_ks['target'] == 1).sum()

cdf_bons = [((df_modelagem_ks['target'] == 0) & (df_modelagem_ks['score'] <= s)).sum() / total_bons
            for s in scores_range]
cdf_maus = [((df_modelagem_ks['target'] == 1) & (df_modelagem_ks['score'] <= s)).sum() / total_maus
            for s in scores_range]
diffs    = [abs(b - m) for b, m in zip(cdf_bons, cdf_maus)]
ks_idx   = diffs.index(max(diffs))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(scores_range, cdf_bons, 'b-o', linewidth=2.5, markersize=7,
        label=f'Bons Pagadores (Adimplentes) — n={total_bons:,}')
ax.plot(scores_range, cdf_maus, 'r-o', linewidth=2.5, markersize=7,
        label=f'Maus Pagadores (Inadimplentes) — n={total_maus:,}')

ax.vlines(scores_range[ks_idx],
          min(cdf_bons[ks_idx], cdf_maus[ks_idx]),
          max(cdf_bons[ks_idx], cdf_maus[ks_idx]),
          color='black', linestyle='--', linewidth=2,
          label=f'KS = {max(diffs):.4f} (score {scores_range[ks_idx]})')

mid_y = (cdf_bons[ks_idx] + cdf_maus[ks_idx]) / 2
ax.annotate(f'  KS = {max(diffs):.4f}',
            xy=(scores_range[ks_idx], mid_y),
            fontsize=13, fontweight='bold', color='black')

ax.fill_between(scores_range, cdf_bons, cdf_maus, alpha=0.08, color='purple')

ax.set_xlabel('Score do Modelo', fontsize=12)
ax.set_ylabel('Frequência Acumulada', fontsize=12)
ax.set_title(f'Curva KS — Janela de Modelagem ({DT_MODELAGEM_INICIO[:7]} a {DT_MODELAGEM_FIM[:7]})\n'
             f'Scores ativos (>{SCORE_MINIMO}) | Apenas contratados', fontsize=13)
ax.set_xticks(scores_range)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

📊 **Leitura da métrica: KS — Poder Discriminatório do Modelo (Pilar 2 — Discriminação)**

**Resultado obtido: KS (Janela de Modelagem DEV) = 0.2554** — classificação: **Aceitável ⚠️**

- **O que a métrica diz:** O KS (Kolmogorov-Smirnov) mede a máxima diferença entre a distribuição acumulada de bons e maus pagadores ao longo das faixas de score. Um KS de 0.2554 significa que, no melhor ponto de corte, o modelo separa 25.5 pontos percentuais a mais de maus do que de bons acumulados.
- **Thresholds de referência (Aguiar et al.):**
  | KS | Interpretação | Status deste modelo |
  |----|---------------|---------------------|
  | > 0.40 | Forte ✅ | — |
  | 0.30 – 0.40 | Bom ✅ | — |
  | 0.20 – 0.30 | Aceitável ⚠️ | **KS = 0.2554 — aqui** |
  | < 0.20 | Fraco ❌ | — |
- **Interpretação:** O KS de 0.2554 situa o modelo na faixa "aceitável" segundo Aguiar et al. O p-valor de 0.0000 confirma que a diferença entre as distribuições é estatisticamente significativa — não é resultado de acaso.
- **Comparação entre períodos:** Este KS foi calculado sobre a **janela de modelagem (DEV: mar/22–fev/23)**. Para diagnóstico completo de monitoramento, é necessário recalcular para o OOT (mar/23–fev/24): se o KS cair abaixo de 0.20 no OOT, o modelo perdeu poder discriminatório relevante fora do tempo.
- **Implicação para o modelo em produção:** Um KS de 0.2554 no DEV indica discriminação moderada. O modelo consegue ordenar razoavelmente bons e maus, mas não de forma excepcional. Isso coloca pressão sobre a calibração — se a separação é modesta, acertar o patamar de inadimplência torna-se ainda mais crítico.
- **Recomendação para a equipe de ciência de dados:** Calcular KS para OOT e para produção (mai/24–dez/24). Uma queda de mais de 5 p.p. entre DEV e OOT já é sinal de degradação relevante. O gráfico de curva KS auxilia na visualização de qual faixa de score concentra o maior poder de separação.
- **Narrativa para apresentação:** "O modelo separa bons e maus pagadores com KS de 25.5% no período de treinamento — dentro da faixa aceitável para crédito. O próximo passo é verificar se essa capacidade se mantém no período de teste fora do tempo."

In [ ]:
# ── KS por período: DEV, OOT e Produção ──────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Filtros base — scores 1-4 descontinuados (Easter Egg #3) excluídos via > 4
df_analise = df[
    (df['contratado'] == 1) &
    (df['score'] > 4) &
    (df['target'].notna())
].copy()

# Janelas temporais
dev  = df_analise[(df_analise['ano_mes'] >= '2022-03-01') & (df_analise['ano_mes'] < '2023-03-01')]
oot  = df_analise[(df_analise['ano_mes'] >= '2023-03-01') & (df_analise['ano_mes'] < '2024-03-01')]
prod = df_analise[(df_analise['ano_mes'] >= '2024-05-01') & (df_analise['ano_mes'] <= '2024-12-31')]

janelas = {'DEV (mar/22-fev/23)': dev, 'OOT (mar/23-fev/24)': oot, 'Producao (mai/24-dez/24)': prod}

print(f"{'Janela':<30} {'N':>8} {'Maus':>8} {'Taxa Inad':>12} {'KS':>8}")
print("-" * 72)

resultados_ks = {}
for nome, df_j in janelas.items():
    # Calcular KS manualmente
    df_sorted = df_j.sort_values('score', ascending=False).reset_index(drop=True)
    total_bons = (df_sorted['target'] == 0).sum()
    total_maus = (df_sorted['target'] == 1).sum()
    
    cum_bons = (df_sorted['target'] == 0).cumsum() / total_bons
    cum_maus = (df_sorted['target'] == 1).cumsum() / total_maus
    
    diff = abs(cum_maus - cum_bons)
    ks_val = diff.max()
    
    taxa = df_j['target'].mean()
    resultados_ks[nome] = ks_val
    print(f"{nome:<30} {len(df_j):>8,} {int(df_j['target'].sum()):>8,} {taxa:>11.1%} {ks_val:>8.4f}")

# ── Curvas KS por janela ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle("Curva KS por Periodo - Distribuicao Acumulada de Bons vs Maus", fontsize=14, fontweight='bold')

cores = {'DEV (mar/22-fev/23)': '#2196F3', 'OOT (mar/23-fev/24)': '#FF9800', 'Producao (mai/24-dez/24)': '#4CAF50'}

for ax, (nome, df_j) in zip(axes, janelas.items()):
    df_sorted = df_j.sort_values('score', ascending=False).reset_index(drop=True)
    total_bons = (df_sorted['target'] == 0).sum()
    total_maus = (df_sorted['target'] == 1).sum()

    cum_bons = (df_sorted['target'] == 0).cumsum() / total_bons
    cum_maus = (df_sorted['target'] == 1).cumsum() / total_maus
    pct_pop  = np.arange(1, len(df_sorted) + 1) / len(df_sorted)

    diff    = abs(cum_maus - cum_bons)
    ks_idx  = diff.idxmax()
    ks_val  = diff[ks_idx]

    cor = cores[nome]
    ax.plot(pct_pop, cum_bons.values, color='steelblue', lw=2, label='Bons (adimplentes)')
    ax.plot(pct_pop, cum_maus.values, color='tomato',    lw=2, label='Maus (inadimplentes)')
    ax.plot(pct_pop, pct_pop,         color='gray',      lw=1, linestyle='--', label='Linha aleatoria')

    ax.axvline(x=pct_pop[ks_idx], color=cor, linestyle=':', lw=1.5)
    ax.annotate(f'KS = {ks_val:.4f}',
                xy=(pct_pop[ks_idx], (cum_bons[ks_idx] + cum_maus[ks_idx]) / 2),
                xytext=(pct_pop[ks_idx] + 0.05, 0.4),
                fontsize=11, fontweight='bold', color=cor,
                arrowprops=dict(arrowstyle='->', color=cor))

    ax.set_title(nome, fontsize=11, fontweight='bold')
    ax.set_xlabel('% da populacao (ordenada por score decrescente)')
    ax.set_ylabel('% acumulada')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Variacao relativa entre periodos ─────────────────────────────────────────
vals = list(resultados_ks.values())
print("\nVariacao do KS entre periodos:")
print(f"  DEV -> OOT:        {vals[1] - vals[0]:+.4f} ({(vals[1]/vals[0]-1)*100:+.1f}%)")
print(f"  OOT -> Producao:   {vals[2] - vals[1]:+.4f} ({(vals[2]/vals[1]-1)*100:+.1f}%)")
print(f"  DEV -> Producao:   {vals[2] - vals[0]:+.4f} ({(vals[2]/vals[0]-1)*100:+.1f}%)")


📊 **Leitura da métrica: KS comparado entre DEV, OOT e Produção**

- **O que a métrica diz:** O KS (Kolmogorov-Smirnov) mede a máxima separação entre a distribuição acumulada de bons pagadores e maus pagadores ao longo da escala de score — quanto maior, mais o modelo consegue distinguir os dois grupos. Segundo Aguiar et al. (2016), valores entre 0,20 e 0,30 são considerados **aceitáveis**, acima de 0,30 são **bons** e acima de 0,40 são **fortes**. Queda superior a 5 p.p. entre janelas consecutivas já configura sinal de degradação relevante.

- **Comparação entre períodos:** O KS calculado na **DEV** representa o poder de discriminação na janela de desenvolvimento do modelo (mar/22–fev/23), que é a referência. Na **OOT** (mar/23–fev/24), observa-se o comportamento fora do tempo — primeiro teste real de generalização. Na **Produção** (mai/24–dez/24), temos o único período com target maduro disponível para validar o modelo em campo real. Uma queda monotônica DEV → OOT → Produção indica degradação progressiva do poder discriminatório; uma recuperação em Produção pode indicar mudança de perfil do público aprovado.

- **Implicação para o modelo em produção:** Se o KS em Produção permanecer acima de 0,20, o modelo ainda possui poder discriminatório mínimo e pode ser mantido em operação com monitoramento intensificado. Se cair abaixo de 0,20 (faixa "Fraco" de Aguiar et al.), a recomendação imediata é retreino — o modelo perdeu capacidade de separar risco de forma confiável. A curva KS também revela *onde* está o poder de separação: se o ponto de máximo KS migrou para faixas de score mais baixas, o corte de aprovação pode precisar ser recalibrado.

- **Recomendação para a equipe de ciência de dados:** Monitorar o KS mensalmente com alerta automático em quedas > 3 p.p. mês a mês. Se a queda DEV → Produção ultrapassar 5 p.p., iniciar análise de qual variável explicativa está com maior CSI — essa variável é a candidata a causar a degradação discriminatória. Caso o KS em Produção seja aceitável mas a curva revele concentração do poder discriminatório em decis intermediários (e não nos extremos), revisar os pontos de corte do policy de crédito.

- **Narrativa para apresentação:** O gráfico mostra se o modelo ainda consegue, na prática, separar quem vai pagar de quem vai dar calote — e se esse poder de separação caiu desde que o modelo foi criado; uma queda relevante aqui é o sinal mais direto de que o modelo precisa ser revisado.


## 2.3. Previsibilidade

**Métricas**:
1. Observado x Esperado

### 2.3.1. Observado x Esperado (GH)

In [ ]:
# GH — Observado × Esperado por faixa de score
# Esperado : taxa de inadimplência observada na janela de modelagem (baseline do modelo)
# Observado: taxa de inadimplência na janela de observação/produção

filtro_esp = (
    (df['ano_mes'] >= DT_MODELAGEM_INICIO) &
    (df['ano_mes'] <= DT_MODELAGEM_FIM) &
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1) &
    (df['target'].notna())
)
filtro_obs = (
    (df['ano_mes'] >= DT_OBSERVACAO_INICIO) &
    (df['ano_mes'] < DT_VAZIO_INICIO) &
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1) &
    (df['target'].notna())
)

gh_esp = (df[filtro_esp]
           .groupby('score')['target']
           .mean()
           .rename('Esperado (Modelagem)'))

gh_obs = (df[filtro_obs]
           .groupby('score')['target']
           .mean()
           .rename('Observado (Producao)'))

gh_df = pd.concat([gh_esp, gh_obs], axis=1).dropna()
gh_df['Desvio (p.p.)'] = (gh_df['Observado (Producao)'] - gh_df['Esperado (Modelagem)']) * 100

# ── Gráfico ─────────────────────────────────────────────────────────────────
x     = np.arange(len(gh_df))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, gh_df['Esperado (Modelagem)'] * 100,
               width, label='Esperado — Modelagem', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, gh_df['Observado (Producao)'] * 100,
               width, label='Observado — Producao/Observacao', color='#E74C3C', alpha=0.8)

# Anotações de desvio em p.p.
for xi, (_, row) in zip(x, gh_df.iterrows()):
    desvio = row['Desvio (p.p.)']
    cor    = '#C0392B' if desvio > 0 else '#27AE60'
    sinal  = '+' if desvio > 0 else ''
    ax.annotate(f'{sinal}{desvio:.1f}p.p.',
                xy=(xi, max(row['Esperado (Modelagem)'], row['Observado (Producao)']) * 100 + 0.2),
                ha='center', va='bottom', fontsize=9, color=cor, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([f'Score {int(s)}' for s in gh_df.index], rotation=30)
ax.set_ylabel('Taxa de Inadimplencia (%)')
ax.set_title(
    'GH — Previsibilidade: Observado x Esperado por Faixa de Score\n'
    f'Modelagem: {DT_MODELAGEM_INICIO[:7]}–{DT_MODELAGEM_FIM[:7]}  |  '
    f'Observacao: {DT_OBSERVACAO_INICIO[:7]}–{DT_VAZIO_INICIO[:7]}',
    fontsize=12
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# ── Tabela resumo do desvio ──────────────────────────────────────────────────
display(
    gh_df.assign(**{
        'Esperado (Modelagem)': gh_df['Esperado (Modelagem)'].map('{:.2%}'.format),
        'Observado (Producao)': gh_df['Observado (Producao)'].map('{:.2%}'.format),
        'Desvio (p.p.)'       : gh_df['Desvio (p.p.)'].map('{:+.2f}'.format),
    })
)

# ── Gráfico de linha por score — estilo professor ────────────────────────────
# Mostra SIMULTANEAMENTE: discriminacao (ranking entre scores) + previsibilidade (estabilidade)
# "se as linhas mantêm a ordem mas sobem → perda de previsibilidade sem perda de discriminacao"
# "se as linhas cruzam mas ficam estáveis → perda de discriminacao com boa previsibilidade"

filtro_linha = (
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1) &
    (df['target'].notna()) &
    (df['ano_mes'] < pd.Timestamp(DT_VAZIO_INICIO))
)
df_linha = df[filtro_linha].copy()

taxa_por_score_tempo = (
    df_linha.groupby(['ano_mes', 'score'])['target']
            .mean()
            .reset_index()
            .rename(columns={'target': 'taxa_inadimplencia'})
)

scores_ativos = sorted(df_linha['score'].unique())
paleta = plt.cm.RdYlGn_r(
    [(i / (len(scores_ativos) - 1)) for i in range(len(scores_ativos))]
)

fig, ax = plt.subplots(figsize=(14, 6))

# Janelas de fundo
janelas_l = {k: pd.Timestamp(v) for k, v in {
    'modelagem_inicio' : DT_MODELAGEM_INICIO,
    'observacao_inicio': DT_OBSERVACAO_INICIO,
    'limbo_inicio'     : DT_LIMBO_INICIO,
    'producao_inicio'  : DT_PRODUCAO_INICIO,
    'vazio_inicio'     : DT_VAZIO_INICIO,
}.items()}
tmax_l = pd.Timestamp(taxa_por_score_tempo['ano_mes'].max())

ax.axvspan(janelas_l['modelagem_inicio'],  janelas_l['observacao_inicio'], color=COR_MODELAGEM,  alpha=0.15, zorder=0)
ax.axvspan(janelas_l['observacao_inicio'], janelas_l['limbo_inicio'],      color=COR_OBSERVACAO, alpha=0.15, zorder=0)
ax.axvspan(janelas_l['limbo_inicio'],      janelas_l['producao_inicio'],   color=COR_LIMBO,      alpha=0.25, zorder=0)
ax.axvspan(janelas_l['producao_inicio'],   tmax_l,                         color=COR_PRODUCAO,   alpha=0.15, zorder=0)

# Uma linha por score
for score, cor in zip(scores_ativos, paleta):
    dados = taxa_por_score_tempo[taxa_por_score_tempo['score'] == score]
    ax.plot(dados['ano_mes'], dados['taxa_inadimplencia'] * 100,
            '-o', color=cor, lw=2, markersize=4, label=f'Score {int(score)}')

ax.set_xlabel('Mes/Ano')
ax.set_ylabel('Taxa de Inadimplencia (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.set_title(
    'GH Temporal — Taxa de Inadimplencia por Score ao Longo do Tempo\n'
    'Linhas separadas = discriminacao mantida  |  Linhas estaveis = boa previsibilidade',
    fontsize=12
)
ax.legend(title='Score', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)

# Rotulos das janelas no topo
for rotulo, x_pos in [
    ('Modelagem',  janelas_l['modelagem_inicio']  + (janelas_l['observacao_inicio'] - janelas_l['modelagem_inicio'])  / 2),
    ('Observacao', janelas_l['observacao_inicio'] + (janelas_l['limbo_inicio']      - janelas_l['observacao_inicio']) / 2),
    ('Limbo',      janelas_l['limbo_inicio']      + (janelas_l['producao_inicio']   - janelas_l['limbo_inicio'])      / 2),
    ('Producao',   janelas_l['producao_inicio']   + (tmax_l                         - janelas_l['producao_inicio'])   / 2),
]:
    ax.text(x_pos, ax.get_ylim()[1] if ax.get_ylim()[1] else 20,
            rotulo, ha='center', va='top', fontsize=8, color='gray', style='italic')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

📊 **Leitura da métrica: Observado vs Esperado por faixa de score — GH (Pilar 3 — Previsibilidade)**

**Resultado: Inadimplência observada MAIOR que esperada em TODAS as faixas — modelo sistematicamente otimista ⚠️**

- **Tabela de resultados:**
  | Score | Esperado (DEV) | Observado (Produção) | Desvio (p.p.) | Razão Obs/Esp |
  |-------|---------------|---------------------|---------------|---------------|
  | 5 | 18.55% | 22.03% | +3.48 | 1.19 ⚠️ |
  | 6 | 13.67% | 16.33% | +2.66 | 1.19 ⚠️ |
  | 7 | 8.85% | 11.13% | +2.28 | 1.26 ❌ |
  | 8 | 5.06% | 6.66% | +1.60 | 1.32 ❌ |
  | 9 | 2.79% | 3.91% | +1.12 | 1.40 ❌ |
  | 10 | 1.38% | 1.74% | +0.37 | 1.26 ❌ |
- **Thresholds de referência (Aguiar et al.):** Razão Obs/Esp entre 0.80 e 1.20 → Aceitável ✅ | Fora desse intervalo → Desvio ⚠️/❌
- **Interpretação crítica:** O modelo **subestima o risco em todas as faixas de score**. Os scores 7, 8, 9 e 10 (que deveriam ser de baixo risco) apresentam razões acima de 1.25 — o modelo previu uma inadimplência muito menor do que a que realmente ocorreu. Apenas os scores 5 e 6 ficaram no limite do aceitável (razão 1.19, ligeiramente abaixo de 1.20).
- **Implicação para o modelo em produção:** A subestimação sistemática do risco significa que a empresa está **sub-provisionando** para inadimplência. Se o modelo diz que clientes com score 9 inadimplem 2.79% e na realidade inadimplem 3.91%, as reservas de crédito são menores do que deveriam ser. O efeito é maior nos scores intermediários-altos (7, 8, 9), onde a concentração de carteira é maior.
- **Recomendação para a equipe de ciência de dados:** Seguindo o fluxo decisório do Prof. Danilo: a previsibilidade está ruim (Obs/Esp > 1.20 em 4 de 6 faixas). O próximo passo é avaliar a estabilidade para determinar se a ação é **calibração** (se PSI < 0.10, estabilidade boa) ou **retreino** (se PSI > 0.10, estabilidade ruim). Com o PSI geral ainda baixo, a calibração é candidata prioritária.
- **Narrativa para apresentação:** "O modelo é sistematicamente otimista — em cada faixa de score, a inadimplência real superou o que o modelo previa. Nos clientes considerados de baixo risco (score alto), o erro chega a 40% acima do esperado, o que compromete o dimensionamento das reservas de crédito da empresa."

### 2.3.2. FPD (First Payment Default)

In [ ]:
filtro_linha = (
    (df['score'] > SCORE_MINIMO) &
    (df['contratado'] == 1) &
    (df['target'].notna()) &
    (df['ano_mes'] < pd.Timestamp(DT_VAZIO_INICIO))
)
df_linha = df[filtro_linha].copy()

fpd_analysis = calcular_fpd(df_linha)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(fpd_analysis['score'].astype(str), fpd_analysis['total_contratos'],
        color='steelblue', alpha=0.55, label='Volume de Contratos')
ax2.plot(fpd_analysis['score'].astype(str), fpd_analysis['taxa_fpd'] * 100,
         'r-o', linewidth=2.5, markersize=8, label='Taxa FPD (%)')

# Anotações de taxa
for _, row in fpd_analysis.iterrows():
    ax2.annotate(f"{row['taxa_fpd']*100:.1f}%",
                 xy=(str(int(row['score'])), row['taxa_fpd']*100),
                 xytext=(0, 8), textcoords='offset points',
                 ha='center', fontsize=9, color='red')

ax1.set_xlabel('Score do Modelo', fontsize=12)
ax1.set_ylabel('Qtd. de Contratos', color='steelblue', fontsize=12)
ax2.set_ylabel('Taxa FPD (%)', color='red', fontsize=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1f}%'))

ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

fig.suptitle('Taxa de FPD por Faixa de Score — Monotonicidade do Modelo\n'
             f'(scores ativos: >{SCORE_MINIMO})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Uso:
print(fpd_analysis)

🥚 **Easter Egg identificado: FPD não é substituto do target de 12 meses**

**Evidência numérica:** Taxa de FPD por score vai de 5.1% (score 5) a 0.9% (score 10), mas o modelo mede inadimplência de 18.6% a 1.4% — diferença de 3–5x. 85.6% dos clientes com FPD=1 têm target=0.

- **O que é:** O FPD (First Payment Default — inadimplência na primeira parcela) está sendo analisado como métrica de previsibilidade do modelo. Contudo, as taxas de FPD são sistematicamente muito menores que as taxas de inadimplência no target de 12 meses em todas as faixas de score:
  | Score | Taxa FPD | Taxa Inadimplência (DEV) | Diferença |
  |-------|----------|--------------------------|-----------|
  | 5 | 5.1% | 18.6% | -13.5 p.p. |
  | 6 | 3.9% | 13.7% | -9.8 p.p. |
  | 7 | 2.9% | 8.9% | -6.0 p.p. |
  | 8 | 1.9% | 5.1% | -3.2 p.p. |
  | 9 | 1.3% | 2.8% | -1.5 p.p. |
  | 10 | 0.9% | 1.4% | -0.5 p.p. |
- **Como foi detectado:** Cruzamento de `FPD == 1` com `target == 0` na base de contratados revela que 85.6% dos clientes com FPD acabam regularizando sua situação antes dos 12 meses — portanto não constam como inadimplentes no target.
- **Implicação para o modelo:** Usar FPD como proxy do target criaria um modelo completamente diferente — estaria prevendo não pagamento da primeira parcela, não inadimplência sustentada. Em produção (jan/25+), o FPD continua sendo registrado (~4.7–5.0%) mesmo sem target maturado — isso pode gerar falsa impressão de que o modelo ainda discrimina quando na verdade não temos target para validar.
- **Recomendação de tratamento:** Manter o FPD como métrica operacional secundária (útil para detectar fraudes ou erros cadastrais), mas **nunca** usá-lo como substituto do target nos cálculos de discriminação e previsibilidade.
- **Narrativa para apresentação:** "O fato de um cliente não pagar a primeira parcela não significa que ele vai inadimplir no modelo de 12 meses — 85% dos casos de FPD foram resolvidos. As taxas de FPD são 3 a 5 vezes menores que as taxas de inadimplência real, confirmando que são fenômenos distintos."

### 2.3.3. WOE (Weight of Evidence) e IV (Information Value)

In [ ]:
# calcular_woe_iv e plot_woe_analysis importadas de utils.py

# Base correta: contratados, target disponível, scores ativos, antes do período vazio
df_woe_base = df[
    (df['contratado'] == 1) &
    (df['target'].notna()) &
    (df['score'] > SCORE_MINIMO) &
    (df['ano_mes'] < DT_VAZIO_INICIO)
].copy()
print(f"Base WOE/IV: {len(df_woe_base):,} registros  ({df_woe_base['target'].mean():.1%} inadimplência)")

woe_score, iv_score = plot_woe_analysis(df_woe_base, 'score')

📊 **Leitura da métrica: WoE e IV por variável — poder preditivo individual (Pilar 2 — Discriminação)**

**Base de cálculo: 469.232 registros | 10.8% de inadimplência | Contratados com target disponível e score > 4**

- **O que a métrica diz:** O WoE (Weight of Evidence — Peso de Evidência) mede, para cada faixa de uma variável, o quanto ela "aponta para" adimplência ou inadimplência. O IV (Information Value — Valor de Informação) agrega o WoE de todas as faixas em um único número que indica o poder preditivo total da variável.
- **Thresholds de referência (Aguiar et al.):**
  | IV | Interpretação |
  |----|---------------|
  | < 0.02 | Variável inútil — sem poder preditivo |
  | 0.02 – 0.10 | Fraco — poder preditivo marginal |
  | 0.10 – 0.30 | Médio — variável útil |
  | > 0.30 | Forte — variável muito preditiva |
- **Contexto da base:** Os 469.232 registros correspondem a toda a base com target disponível (contratados, score > 4, target não nulo). Taxa de inadimplência de 10.8% é consistente com o mercado de financiamento de veículos — acima do esperado quando comparado à taxa DEV (calculada isoladamente no DEV a 12–15%), o que antecipa o desvio de previsibilidade visto no GH.
- **O que observar nas curvas de WoE para o score:**
  - A curva de WoE do score deve ser **monotônica decrescente** (score mais alto → WoE mais positivo → menos inadimplentes): se isso se confirma, o score discrimina bem.
  - A inclinação da curva indica quão diferente é o risco entre scores consecutivos. Uma curva plana indica pouca diferenciação entre faixas.
- **Implicação para o modelo em produção:** O IV do score calculado nesta base cobre DEV + OOT + Produção com target (até dez/24) — é uma medida consolidada. Para monitoramento, o IV deve ser recalculado separadamente por janela para detectar degradação temporal do poder preditivo.
- **Recomendação para a equipe de ciência de dados:** Priorizar o monitoramento das variáveis com IV > 0.10 na base de DEV. São essas que mais contribuem para o KS e cujo desvio de comportamento mais impacta a discriminação.
- **Narrativa para apresentação:** "Com 469 mil contratos e 10.8% de inadimplência — uma taxa já acima do esperado pelo modelo — calculamos o poder preditivo de cada variável. O resultado nos diz quais características dos clientes ainda estão ajudando o modelo a distinguir bons de maus pagadores."

---

# 3. Síntese do Monitoramento e Recomendação Final

## 3.1. Resumo dos 3 Pilares

### Pilar 1 — Estabilidade

| Métrica | Resultado | Classificação |
|---------|-----------|---------------|
| PSI do Score (DEV vs dez/24, base inadimplentes) | 0.0019 | **Estável ✅** |
| Carta de Controle por Score Individual | Nenhum score cruzou limites | **Estável ✅** |
| Carta de Controle — Média Ponderada | mai–set/24 abaixo do LI (mín. 6.264 vs LI 6.298) | **Atenção ⚠️** |

**Veredito Estabilidade:** O perfil da população inadimplente está majoritariamente estável. A queda temporária da média ponderada do score (mai–set/24) foi recuperada, mas coincide com o início da janela de produção — requer monitoramento contínuo. O PSI oficial sobre a **população total contratada** ainda precisa ser calculado para completar o pilar.

---

### Pilar 2 — Discriminação

| Métrica | Resultado | Classificação |
|---------|-----------|---------------|
| KS (Janela de Modelagem DEV) | 0.2554 | **Aceitável ⚠️** |
| KS (OOT) | A calcular | — |
| KS (Produção mai/24–dez/24) | A calcular | — |

**Veredito Discriminação:** O KS de 0.2554 no DEV situa o modelo na faixa aceitável (0.20–0.30) segundo Aguiar et al. O modelo não é excepcional na separação de bons e maus pagadores — o que é esperado dado que o score é discreto (valores 5 a 10), limitando a granularidade da separação. A comparação com o OOT é fundamental e ainda não calculada.

---

### Pilar 3 — Previsibilidade

| Score | Esperado (DEV) | Observado (OOT/Prod) | Razão | Classificação |
|-------|---------------|---------------------|-------|---------------|
| 5 | 18.55% | 22.03% | 1.19 | Limite ⚠️ |
| 6 | 13.67% | 16.33% | 1.19 | Limite ⚠️ |
| 7 | 8.85% | 11.13% | 1.26 | **Desvio ❌** |
| 8 | 5.06% | 6.66% | 1.32 | **Desvio ❌** |
| 9 | 2.79% | 3.91% | 1.40 | **Desvio ❌** |
| 10 | 1.38% | 1.74% | 1.26 | **Desvio ❌** |

**Veredito Previsibilidade:** O modelo **subestima o risco em todas as faixas de score**. A inadimplência real supera a prevista em 19–40%, com os scores de menor risco (7, 8, 9, 10) apresentando os maiores desvios relativos. Isso compromete o dimensionamento de provisões para inadimplência.

---

## 3.2. Fluxo de Decisão (Prof. Danilo — Slide 18)

```
DISCRIMINAÇÃO ruim (KS < 0.20)? → NÃO (KS = 0.2554 — aceitável)
       ↓
PREVISIBILIDADE ruim (Obs/Esp > 1.20)? → SIM (scores 7, 8, 9, 10 fora do intervalo)
       ↓
ESTABILIDADE boa (PSI < 0.10)? → SIM (PSI = 0.0019 — estável)
       ↓
AÇÃO RECOMENDADA: CALIBRAR O MODELO
```

**Conclusão:** Com discriminação aceitável, previsibilidade comprometida e estabilidade boa, o fluxo decisório aponta para **calibração** — ajuste dos limiares de probabilidade e dos thresholds de score sem necessidade de retreino completo. A calibração deve corrigir a subestimação sistemática de risco antes que o cohort do spike de aprovação (out/24–mar/25) complete maturação (out/25–mar/26).

---

## 3.3. Plano de Ação para os 10 Easter Eggs

| # | Easter Egg | Status | Ação Prioritária |
|---|-----------|--------|-----------------|
| 1 | `vl_bem = '25a70'` | ✅ Tratado no notebook | Corrigir na origem (sistema de cadastro) |
| 2 | 347 IDs duplicados | ⚠️ Identificado | Criar chave composta ID+data; auditar pipeline |
| 3 | Scores 1–4 descontinuados | ✅ Tratado (filtro `score > 4`) | Documentar descontinuação; remover da base histórica |
| 4 | Target válido só até dez/24 | ✅ Tratado (flag `target_disponivel`) | Revisar automaticamente a cada ciclo mensal |
| 5 | Viés de sobrevivência jan/25+ | ✅ Tratado (target → NaN) | Análise por safra (vintage) para 2025 |
| 6 | Limbo mar/24–abr/24 | ✅ Tratado (excluído) | Documentar e manter exclusão permanente |
| 7 | `qt_restr` e `idade_veiculo` como string | ⚠️ Identificado | `pd.to_numeric(..., errors='coerce')` no pipeline |
| 8 | FPD ≠ target (85.6% inconsistentes) | ⚠️ Identificado | Separar monitoramento de FPD do monitoramento de inadimplência |
| 9 | Spike aprovação +10 p.p. out/24–mar/25 | ⚠️ Identificado | Investigar causa com time de negócio; cohort separado |
| 10 | Queda -40% volume a partir abr/25 | ⚠️ Identificado | Normalizar volumes no PSI; documentar no relatório |

---

## 3.4. Classificação Tier I e Cadência de Monitoramento

**Este modelo é Tier I** (crédito = alto impacto em negócios E em clientes):

> "Este modelo se enquadra como **Tier I** (alto impacto em negócios e clientes), portanto exige acompanhamento completo dos 3 pilares com granularidade por variável em frequência **semanal/mensal**."

**Cadência recomendada:**
- **Semanal:** Carta de controle do score médio; taxa de aprovação; volume de propostas
- **Mensal:** PSI + CSI por variável; KS; Obs vs Esperado por faixa; alerta de spike
- **Trimestral:** Revisão completa dos 3 pilares; avaliação da necessidade de calibração ou retreino
- **Urgente:** Quando PSI > 0.10 **ou** cohort out/24–mar/25 completar 12 meses (out–mar/26) — reprocessar GH com dados maturados

---

## 3.5. Próximos Passos Técnicos

1. **Calcular PSI sobre população total contratada** (`df_contratado`) — o PSI atual é sobre inadimplentes apenas
2. **Calcular KS para OOT e produção** separadamente para ver trajetória de degradação
3. **Calcular CSI por variável** para identificar qual característica da população mudou (especialmente para o período mai–set/24 onde a média do score caiu)
4. **Calibrar o modelo** — ajustar probabilidades previstas para corrigir a razão Obs/Esp para próximo de 1.0 em todas as faixas
5. **Análise de safra (vintage)** para os cohorts de out/24–mar/25 quando target estiver disponível
6. **Corrigir tipagem** de `qt_restr` e `idade_veiculo` no pipeline de dados